In [ ]:
from pathlib import Path
import json

from PIL import Image, ImageOps
from torchvision import transforms

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

split_manifest = json.loads(
    (PROJECT_ROOT / "reports" / "base_split.json").read_text(
        encoding="utf-8"
    )
)
print("Sources:", split_manifest["sources"])
print("Split:", len(split_manifest["training_paths"]), len(split_manifest["validation_paths"]))

IMAGE_SIZE = 128

def resize_with_padding(image):
    return ImageOps.pad(
        image.convert("RGB"),
        (IMAGE_SIZE, IMAGE_SIZE),
        method=Image.Resampling.BILINEAR,
        color=(0, 0, 0),
    )

base_transform = transforms.Compose([
    transforms.Lambda(resize_with_padding),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    ),
])

sample_path = PROJECT_ROOT / split_manifest["training_paths"][0]

with Image.open(sample_path) as image:
    sample_tensor = base_transform(image)

print("Input tensor shape:", sample_tensor.shape)
print("Pixel range:", sample_tensor.min().item(), sample_tensor.max().item())

In [ ]:
with Image.open(sample_path) as image:
    prepared = resize_with_padding(image)

    print("Original:", image.size)
    print("Prepared:", prepared.size)
    display(prepared)

In [ ]:
from torch.utils.data import Dataset

class ManifestImageDataset(Dataset):
    """One class mapping and eligible sample list across all source folders."""
    def __init__(self, manifest, transform=None):
        self.transform = transform
        self.class_to_idx = manifest["class_to_idx"]
        self.classes = sorted(self.class_to_idx, key=self.class_to_idx.get)
        self.samples = [
            (str(PROJECT_ROOT / row["path"]), self.class_to_idx[row["label"]])
            for row in manifest["samples"]
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        with Image.open(path) as image:
            image = image.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

train_dataset = ManifestImageDataset(split_manifest)
print("Eligible images:", len(train_dataset))
print("Class mapping:", train_dataset.class_to_idx)

In [ ]:
path_to_index = {
    Path(path).relative_to(PROJECT_ROOT).as_posix(): index
    for index, (path, label) in enumerate(train_dataset.samples)
}
train_indices = [path_to_index[path] for path in split_manifest["training_paths"]]
val_indices = [path_to_index[path] for path in split_manifest["validation_paths"]]
assert train_indices and val_indices
assert not set(train_indices) & set(val_indices)
assert set(train_indices) | set(val_indices) == set(range(len(train_dataset)))
print("Training indices:", len(train_indices))
print("Validation indices:", len(val_indices))

In [ ]:
from torch.utils.data import Subset

# Separate transforms can be applied later to this same eligible sample list.
train_full = ManifestImageDataset(split_manifest, transform=base_transform)
validation_full = ManifestImageDataset(split_manifest, transform=base_transform)
assert train_dataset.samples == train_full.samples == validation_full.samples
train_subset = Subset(train_full, train_indices)
validation_subset = Subset(validation_full, val_indices)
print("Training images:", len(train_subset))
print("Validation images:", len(validation_subset))
print("Classes:", train_full.classes)
image_tensor, label_index = train_subset[0]
print("One image shape:", image_tensor.shape)
print("Its class:", train_full.classes[label_index])

In [ ]:
import torch
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(42),
)

validation_loader = DataLoader(
    validation_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

batch_images, batch_labels = next(iter(train_loader))

print("Images shape:", batch_images.shape)
print("Labels shape:", batch_labels.shape)
print("First five labels:", batch_labels[:5])


In [ ]:
from torch import nn

# Conv2d output per spatial axis (dilation=1): floor((N + 2*P - K) / S) + 1.
# Here N=128, P=1, K=3, S=1: floor((128 + 2*1 - 3) / 1) + 1 = 128.
# 16 filters make 16 feature maps of 128x128; MaxPool2d(2) makes them 64x64.
first_block = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
)

first_output = first_block(batch_images)

print("Input shape:", batch_images.shape)
print("Output shape:", first_output.shape)

In [ ]:
# Here N=64, P=1, K=3, S=1: floor((64 + 2*1 - 3) / 1) + 1 = 64.
# 32 filters make 32 feature maps of 64x64; MaxPool2d(2) makes them 32x32.
second_block = nn.Sequential(
    nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
)

second_output = second_block(first_output)

print("First block output:", first_output.shape)
print("Second block output:", second_output.shape)

In [ ]:
flatten = nn.Flatten(start_dim=1)
flat_features = flatten(second_output)

print("Before flatten:", second_output.shape)
print("After flatten:", flat_features.shape)

In [ ]:
num_classes = len(train_full.classes)

classifier = nn.Linear(
    in_features=32 * 32 * 32,
    out_features=num_classes,
)

class_scores = classifier(flat_features)

print("Number of classes:", num_classes)
print("Scores shape:", class_scores.shape)
print("First image scores:", class_scores[0])

In [ ]:
first_image_scores = class_scores[0]

for class_index, score in enumerate(first_image_scores):
    class_name = train_full.classes[class_index]
    print(f"{class_index}: {class_name:10s} | score: {score.item():.4f}")

predicted_index = first_image_scores.argmax().item()
target_index = batch_labels[0].item()

print("Predicted class:", train_full.classes[predicted_index])
print("True class:", train_full.classes[target_index])

In [ ]:
loss_fn = nn.CrossEntropyLoss()
batch_loss = loss_fn(class_scores, batch_labels)

per_image_losses = nn.functional.cross_entropy(
    class_scores,
    batch_labels,
    reduction="none",
)

print("Scores shape:", class_scores.shape)
print("Labels shape:", batch_labels.shape)
print("First image loss:", per_image_losses[0].item())
print("Mean of image losses:", per_image_losses.mean().item())
print("Batch loss:", batch_loss.item())

In [ ]:
layers = {
    "first conv": first_block[0],
    "second conv": second_block[0],
    "classifier": classifier,
}

for name, layer in layers.items():
    print(name, "gradient before backward:", layer.weight.grad is None)

batch_loss.backward()

for name, layer in layers.items():
    print(
        name,
        "gradient shape:", tuple(layer.weight.grad.shape),
        "mean absolute gradient:", layer.weight.grad.abs().mean().item(),
    )

In [ ]:
all_parameters = [
    *first_block.parameters(),
    *second_block.parameters(),
    *classifier.parameters(),
]

optimizer = torch.optim.SGD(all_parameters, lr=0.01)

old_loss = batch_loss.item()
old_classifier_weights = classifier.weight.detach().clone()

optimizer.step()

with torch.no_grad():
    new_scores = classifier(
        flatten(
            second_block(
                first_block(batch_images)
            )
        )
    )
    new_loss = loss_fn(new_scores, batch_labels)

largest_weight_change = (
    classifier.weight.detach() - old_classifier_weights
).abs().max().item()

print("Old loss:", old_loss)
print("New loss:", new_loss.item())
print("Largest classifier weight change:", largest_weight_change)

In [ ]:
print("Before clearing:", classifier.weight.grad is None)
print("Gradient size:", classifier.weight.grad.abs().mean().item())

optimizer.zero_grad(set_to_none=True)

for name, layer in layers.items():
    print(name, "gradient after clearing:", layer.weight.grad)

In [ ]:
class VehicleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(32 * 32 * 32, num_classes),
        )

    def forward(self, images):
        features = self.features(images)
        scores = self.classifier(features)
        return scores


torch.manual_seed(42)
model = VehicleCNN(num_classes=len(train_full.classes))

print(model)
print("Output shape:", model(batch_images).shape)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = model.to(device)

LEARNING_RATE = 1e-3

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

print("Device:", device)
print("Learning rate:", LEARNING_RATE)
print("Optimizer:", type(optimizer).__name__)

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        scores = model(images)
        loss = loss_fn(scores, labels)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size

        predicted_indices = scores.argmax(dim=1)
        correct_predictions += (
            predicted_indices == labels
        ).sum().item()

        total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }

In [ ]:
def evaluate(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            scores = model(images)
            loss = loss_fn(scores, labels)

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size

            predicted_indices = scores.argmax(dim=1)
            correct_predictions += (
                predicted_indices == labels
            ).sum().item()

            total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }


In [ ]:
initial_validation = evaluate(
    model,
    validation_loader,
    loss_fn,
    device,
)

print("Initial validation loss:", initial_validation["loss"])
print(
    "Initial validation accuracy:",
    f'{initial_validation["accuracy"]:.1%}',
)

In [ ]:
from copy import deepcopy

# Start this experiment with fresh weights and a fixed shuffle seed.
torch.manual_seed(42)
train_loader.generator.manual_seed(42)

model = VehicleCNN(
    num_classes=len(train_full.classes)
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

EPOCHS = 30
history = []

best_val_accuracy = -1.0
best_epoch = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, loss_fn, device
    )
    val_metrics = evaluate(
        model, validation_loader, loss_fn, device
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
    })

    if val_metrics["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_metrics["accuracy"]
        best_epoch = epoch
        best_state = deepcopy(model.state_dict())

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'train loss: {train_metrics["loss"]:.4f} | '
        f'train acc: {train_metrics["accuracy"]:.1%} | '
        f'val loss: {val_metrics["loss"]:.4f} | '
        f'val acc: {val_metrics["accuracy"]:.1%}'
    )

print(
    f"Best validation accuracy: {best_val_accuracy:.1%} "
    f"at epoch {best_epoch}"
)

## تفسیر نتیجهٔ CNN پایه

پس از اجرای سلول آموزش، از `history` بهترین epoch، دقت و loss آموزش و validation را بخوان. فاصلهٔ زیاد دقت آموزش و validation می‌تواند نشانهٔ بیش‌برازش باشد. عددهای اجرای قبلی به تقسیم قبلی مربوط‌اند؛ این بخش پس از آموزش روی تقسیم یکپارچه باید با خروجی تازه تکمیل شود.


In [ ]:
checkpoint_dir = PROJECT_ROOT / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "cnn_baseline_best.pt"

checkpoint = {
    "architecture": "VehicleCNN",
    "model_state_dict": {
        name: tensor.detach().cpu()
        for name, tensor in best_state.items()
    },
    "class_to_idx": train_full.class_to_idx,
    "best_epoch": best_epoch,
    "best_val_accuracy": best_val_accuracy,
    "preprocessing": {
        "image_size": IMAGE_SIZE,
        "resize": "aspect-ratio-preserving padding",
        "interpolation": "bilinear",
        "padding_color": [0, 0, 0],
        "mean": [0.5, 0.5, 0.5],
        "std": [0.5, 0.5, 0.5],
    },
    "training_config": {
        "seed": 42,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "loss": "CrossEntropyLoss",
    },
    "split_fingerprint": split_manifest["fingerprint"],
    "history": history,
    "review_threshold": None,  # Not calibrated yet.
}

# This is a binary file; do not edit or resave it as text.
torch.save(checkpoint, checkpoint_path)

# Verify that the saved checkpoint can be read immediately.
verified_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)
assert verified_checkpoint["best_epoch"] == best_epoch
assert verified_checkpoint["class_to_idx"] == train_full.class_to_idx
print("Checkpoint read-back verified.")

print("Saved to:", checkpoint_path)
print("Best epoch:", checkpoint["best_epoch"])
print("Validation accuracy:", f'{best_val_accuracy:.1%}')
# Compare checkpoint selection with the lowest validation loss in this same run.
lowest_loss_row = min(
    verified_checkpoint["history"],
    key=lambda row: row["val_loss"],
)
print("Checkpoint selected by: highest validation accuracy")
print("Lowest validation loss:", f'{lowest_loss_row["val_loss"]:.4f}',
      "at epoch", lowest_loss_row["epoch"])
print("Validation accuracy at lowest loss:",
      f'{lowest_loss_row["val_accuracy"]:.1%}')


In [ ]:
import matplotlib.pyplot as plt

epochs = [row["epoch"] for row in history]
train_losses = [row["train_loss"] for row in history]
val_losses = [row["val_loss"] for row in history]

train_accuracies = [
    100 * row["train_accuracy"] for row in history
]
val_accuracies = [
    100 * row["val_accuracy"] for row in history
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, marker="o", label="Train")
axes[0].plot(epochs, val_losses, marker="o", label="Validation")
axes[0].set_title("Cross-entropy loss")
axes[0].set_ylabel("Loss")

axes[1].plot(epochs, train_accuracies, marker="o", label="Train")
axes[1].plot(epochs, val_accuracies, marker="o", label="Validation")
axes[1].set_title("Classification accuracy")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_ylim(0, 100)

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.set_xticks(epochs)
    ax.axvline(
        best_epoch,
        linestyle="--",
        color="gray",
        label="Selected epoch",
    )
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()

figure_dir = PROJECT_ROOT / "reports" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(
    figure_dir / "cnn_baseline_curves.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

In [ ]:
checkpoint_path = PROJECT_ROOT / "checkpoints" / "cnn_baseline_best.pt"

# Recover a checkpoint accidentally resaved as UTF-16 text.
with checkpoint_path.open("rb") as checkpoint_file:
    header = checkpoint_file.read(2)

if header in (b"\xfe\xff", b"\xff\xfe"):
    if "checkpoint" not in globals():
        raise RuntimeError(
            "The checkpoint was converted to text. "
            "Run the checkpoint save cell using best_state still in memory."
        )
    assert checkpoint["class_to_idx"] == train_full.class_to_idx
    torch.save(checkpoint, checkpoint_path)
    print("Restored binary checkpoint from the in-memory checkpoint dictionary.")

saved_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

assert (
    saved_checkpoint["class_to_idx"]
    == train_full.class_to_idx
)

# Current checkpoints store only the fingerprint; accept older files for the check.
saved_split_fingerprint = saved_checkpoint.get("split_fingerprint")
if saved_split_fingerprint is None:
    saved_split_fingerprint = saved_checkpoint.get("split_manifest", {}).get("fingerprint")
if saved_split_fingerprint != split_manifest["fingerprint"]:
    raise RuntimeError("Checkpoint belongs to a different data split. Train and save with the current split first.")

best_model = VehicleCNN(
    num_classes=len(saved_checkpoint["class_to_idx"])
).to(device)

best_model.load_state_dict(
    saved_checkpoint["model_state_dict"]
)

best_validation = evaluate(
    best_model,
    validation_loader,
    loss_fn,
    device,
)

print("Loaded epoch:", saved_checkpoint["best_epoch"])
print("Validation loss:", best_validation["loss"])
print("Validation accuracy:", f'{best_validation["accuracy"]:.1%}')

In [ ]:
best_model.eval()

y_true = []
y_pred = []
probability_batches = []

with torch.no_grad():
    for images, labels in validation_loader:
        scores = best_model(images.to(device))
        probabilities = torch.softmax(scores, dim=1)
        predictions = scores.argmax(dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(predictions.cpu().tolist())
        probability_batches.append(probabilities.cpu())

val_probabilities = torch.cat(probability_batches, dim=0)

correct = sum(
    true_label == predicted_label
    for true_label, predicted_label in zip(y_true, y_pred)
)

print("Number of images:", len(y_true))
print("Probability shape:", val_probabilities.shape)
print("Correct predictions:", correct)
print("First ten true labels:", y_true[:10])
print("First ten predictions:", y_pred[:10])

In [ ]:
from sklearn.metrics import classification_report

class_names = train_full.classes
class_indices = list(range(len(class_names)))
report_text = classification_report(
    y_true, y_pred, labels=class_indices, target_names=class_names,
    digits=3, zero_division=0,
)
print(report_text)

baseline_report = classification_report(
    y_true, y_pred, labels=class_indices, target_names=class_names,
    output_dict=True, zero_division=0,
)
baseline_report.update({
    "experiment": "cnn_baseline",
    "split_fingerprint": split_manifest["fingerprint"],
    "result_status": "current_split",
    "training_images": len(train_subset),
    "validation_images": len(validation_subset),
    "best_epoch": best_epoch,
})
report_path = PROJECT_ROOT / "reports/experiment_results.json"
all_results = json.loads(report_path.read_text()) if report_path.exists() else {}
all_results["_split"] = {
    "current_fingerprint": split_manifest["fingerprint"],
    "comparison_rule": "Compare results only when split_fingerprint matches.",
}
all_results["cnn_baseline"] = baseline_report
report_path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


## تحلیل هر کلاس روی validation فعلی

گزارش سلول قبل برای هر کلاس `precision`، `recall`، `F1` و تعداد نمونه‌های واقعی (`support`) را نشان می‌دهد. کمترین precision یعنی پیش‌بینی‌های مثبتِ اشتباه بیشتر است؛ کمترین recall یعنی نمونه‌های واقعی بیشتری از دست رفته‌اند. برای نتیجه‌گیری دربارهٔ علت خطا، ماتریس خطا و خود تصاویر را هم بررسی کن. بخشی از برچسب‌های `unclean` در این validation هنوز موقت‌اند.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_counts = confusion_matrix(
    y_true, y_pred, labels=class_indices
)

cm_normalized = confusion_matrix(
    y_true, y_pred, labels=class_indices, normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ConfusionMatrixDisplay(
    cm_counts, display_labels=class_names
).plot(
    ax=axes[0],
    cmap="Blues",
    values_format="d",
    xticks_rotation=45,
    colorbar=False,
)

ConfusionMatrixDisplay(
    cm_normalized, display_labels=class_names
).plot(
    ax=axes[1],
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    colorbar=False,
)

axes[0].set_title("Validation — image counts")
axes[1].set_title("Validation — normalized by true class")

plt.tight_layout()
plt.savefig(
    figure_dir / "cnn_baseline_confusion_matrices.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## ماتریس خطای جداگانه برای هر کلاس

برای هر کلاس، مسئله را موقتاً به «این کلاس» و «سایر کلاس‌ها» تبدیل می‌کنیم. در هر ماتریس ۲×۲، **سطر برچسب واقعی** و **ستون پیش‌بینی** است:

| | پیش‌بینیِ سایر | پیش‌بینیِ کلاس |
|---|---:|---:|
| واقعیِ سایر | TN | FP |
| واقعیِ کلاس | FN | TP |

برای مثال، تصویری که واقعاً `kamyun` است ولی `kamyunet` پیش‌بینی شده، برای `kamyun` یک FN و برای `kamyunet` یک FP است. ماتریس ۸×۸ قبلی همچنان برای دیدن *جفت کلاس‌های اشتباه‌شده* لازم است؛ این هشت ماتریس برای محاسبهٔ TP/FP/FN/TN هر کلاس‌اند.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for class_index, (class_name, ax) in enumerate(zip(class_names, axes.flat)):
    # One-versus-rest: True means "this class", False means "all other classes".
    actual_is_class = [label == class_index for label in y_true]
    predicted_is_class = [label == class_index for label in y_pred]
    binary_cm = confusion_matrix(
        actual_is_class, predicted_is_class, labels=[False, True]
    )
    tn, fp, fn, tp = binary_cm.ravel()
    assert tn + fp + fn + tp == len(y_true)
    print(f"{class_name:10} | TP={tp:3d} FP={fp:3d} FN={fn:3d} TN={tn:3d}")

    ConfusionMatrixDisplay(
        binary_cm, display_labels=["Other", class_name]
    ).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(class_name)

plt.tight_layout()
plt.show()


## بررسی خطاهای با اطمینان بالا

در ماتریس خطا، سطر برچسب واقعی و ستون پیش‌بینی مدل است. ۱۳ کامیون به کامیونت و ۹ کامیونت به کامیون اشتباه شده‌اند. ۸ کامیونت نیز آمبولانس پیش‌بینی شده‌اند. این اعداد به‌تنهایی علت اشتباه را ثابت نمی‌کنند؛ باید تصاویر را بررسی کنیم.

سلول بعدی تمام موارد اشتباه را پیدا می‌کند، آن‌ها را بر اساس احتمالِ کلاس پیش‌بینی‌شده از زیاد به کم مرتب می‌کند و ۱۲ مورد اول را نشان می‌دهد. این مقدار، اطمینان خروجی softmax است، نه احتمال تضمین‌شدهٔ درست‌بودن پاسخ. اگر برچسبی مشکوک بود، فایل خام را تغییر نده؛ ابتدا آن را در فهرست بازبینی محلی ثبت کنیم.


In [ ]:
# The validation DataLoader uses shuffle=False, so these positions match y_true/y_pred.
wrong_indices = [
    i for i, (true_label, predicted_label) in enumerate(zip(y_true, y_pred))
    if true_label != predicted_label
]
wrong_indices.sort(
    key=lambda i: val_probabilities[i, y_pred[i]].item(),
    reverse=True,
)

print("Validation mistakes:", len(wrong_indices), "out of", len(y_true))
print("Highest-confidence mistakes shown:", min(12, len(wrong_indices)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, position in zip(axes.flat, wrong_indices[:12]):
    # Subset index -> full dataset index -> original image path.
    full_index = validation_subset.indices[position]
    image_path, _ = validation_full.samples[full_index]
    confidence = val_probabilities[position, y_pred[position]].item()

    with Image.open(image_path) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(
        f"True: {class_names[y_true[position]]} | "
        f"Pred: {class_names[y_pred[position]]}\n"
        f"Model confidence: {confidence:.1%}", fontsize=9
    )
    ax.axis("off")
for ax in list(axes.flat)[min(12, len(wrong_indices)):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
position = wrong_indices[0]
full_index = validation_subset.indices[position]
image_path, _ = validation_full.samples[full_index]

print("Image:", image_path)
print("True:", class_names[y_true[position]])
print("Predicted:", class_names[y_pred[position]])

with Image.open(image_path) as image:
    display(image.copy())

## کدام دو کلاس بیشتر با هم اشتباه می‌شوند؟

طبق صورت پروژه، برای هر جفت کلاس، خطای دو جهت را از ماتریسِ نرمال‌شده با سطر واقعی جمع می‌کنیم:

```text
pair_confusion(i, j) = C_normalized[i, j] + C_normalized[j, i]
```

عدد اول می‌گوید چه سهمی از تصاویر واقعی کلاس `i` به اشتباه `j` پیش‌بینی شده‌اند؛ عدد دوم جهت عکس را نشان می‌دهد. این *امتیاز رتبه‌بندیِ جفت‌ها* است، نه درصد دقت مدل یا احتمال یک تصویر. بالا بودنش دلیل کافی برای ادغام کلاس‌ها یا جابه‌جایی فایل‌ها نیست؛ باید تصویر، تعریف کلاس و هزینهٔ خطا را هم بررسی کنیم.


In [ ]:
pair_results = []
for first in range(len(class_names)):
    for second in range(first + 1, len(class_names)):
        first_to_second = cm_normalized[first, second]
        second_to_first = cm_normalized[second, first]
        pair_results.append((
            first_to_second + second_to_first,
            first, second,
            first_to_second, second_to_first,
        ))

pair_results.sort(reverse=True)
for rank, (score, first, second, first_to_second, second_to_first) in enumerate(pair_results[:5], start=1):
    print(
        f"{rank}. {class_names[first]} ↔ {class_names[second]} | "
        f"{class_names[first]}→{class_names[second]}: "
        f"{cm_counts[first, second]}/{cm_counts[first].sum()} = {first_to_second:.3f} | "
        f"{class_names[second]}→{class_names[first]}: "
        f"{cm_counts[second, first]}/{cm_counts[second].sum()} = {second_to_first:.3f} | "
        f"pair score: {score:.3f}"
    )


## ResNet18، قدم ۱: معماری مدل ازپیش‌آموزش‌دیده

طبق صورت پروژه، باید یک ResNet18 با وزن‌های آموزش‌دیده روی ImageNet را برای **استخراج ویژگی** و سپس **fine-tuning لایهٔ `layer4`** مقایسه کنیم. فعلاً فقط مدل را می‌سازیم و لایهٔ ورودی و لایهٔ آخرش را می‌بینیم؛ این سلول **هیچ آموزشی انجام نمی‌دهد** و وزن‌ها را تغییر نمی‌دهد. فایل وزن ImageNet در کش محلی موجود است.

سؤال کلیدی: لایهٔ `fc` فعلی ۵۱۲ ویژگی ورودی می‌گیرد و ۱۰۰۰ score می‌دهد. این ۱۰۰۰ خروجی برای مسئلهٔ ما نیستند؛ در قدم بعد سرِ مدل را با یک لایهٔ ۸خروجی جایگزین می‌کنیم. طبق صورت پروژه، `conv1` و `maxpool` اصلی را برای مدل pretrained دست‌نخورده می‌گذاریم و از preprocessing سازگار با ImageNet استفاده می‌کنیم.


In [ ]:
import json
from pathlib import Path
from torchvision import models

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
resnet_manifest = json.loads((project_root / "reports/base_split.json").read_text())

resnet_weights = models.ResNet18_Weights.DEFAULT
resnet18_model = models.resnet18(weights=resnet_weights)

print("First convolution:", resnet18_model.conv1)
print("Last layer:", resnet18_model.fc)
print("Features entering fc:", resnet18_model.fc.in_features)
print("Original output scores:", resnet18_model.fc.out_features)
print("Our vehicle classes:", len(resnet_manifest["class_to_idx"]))
print("ImageNet preprocessing:", resnet_weights.transforms())


## ResNet18، قدم ۲: backbone ثابت، سرِ جدید قابل‌آموزش

ترتیب این سه کار مهم است و از فایل تدریس استاد پیروی می‌کند: **(۱) بارگذاری وزن pretrained، (۲) فریز کردن پارامترهای موجود، (۳) جایگزین کردن `fc` با یک لایهٔ تازهٔ ۸خروجی**. اگر لایهٔ تازه را قبل از فریز کردن بسازیم، آن هم فریز می‌شود. لایهٔ تازه به‌صورت پیش‌فرض `requires_grad=True` دارد.

در این مرحله فقط مدل را آماده می‌کنیم؛ هنوز هیچ batch، loss، `backward()` یا `optimizer.step()` نداریم. backbone در forward همچنان ویژگی تولید می‌کند، اما وزن‌هایش در آموزشِ مرحلهٔ اول تغییر نمی‌کنند. انتظار داریم تعداد پارامترهای قابل‌آموزش فقط `512 × 8 + 8 = 4104` باشد: برای هر یک از ۸ کلاس، ۵۱۲ وزن و یک bias. `eval()` کار فریز کردن را انجام نمی‌دهد؛ رفتار BatchNorm را در قدم‌های بعد جداگانه تنظیم می‌کنیم.


In [ ]:
from torch import nn

# Same pretrained architecture as the inspection cell above.
feature_model = models.resnet18(weights=resnet_weights)

# Freeze every parameter that came with the pretrained model.
feature_model.requires_grad_(False)

num_features = feature_model.fc.in_features
num_classes = len(resnet_manifest["class_to_idx"])
feature_model.fc = nn.Linear(num_features, num_classes)

trainable_names = [
    name for name, parameter in feature_model.named_parameters()
    if parameter.requires_grad
]
total_parameters = sum(parameter.numel() for parameter in feature_model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in feature_model.parameters()
    if parameter.requires_grad
)

assert trainable_names == ["fc.weight", "fc.bias"]
assert trainable_parameters == num_features * num_classes + num_classes
print("New classifier:", feature_model.fc)
print("Backbone conv1 trainable:", feature_model.conv1.weight.requires_grad)
print("Classifier weight trainable:", feature_model.fc.weight.requires_grad)
print("Trainable parameter names:", trainable_names)
print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)


## ResNet18، قدم ۳: تصویر را برای وزن‌های ImageNet آماده کنیم

در CNN خودمان تصویر را با padding به `128×128` رساندیم و با میانگین/انحراف معیار `0.5` نرمال کردیم. وزن‌های آمادهٔ ResNet18 با روش دیگری آموزش دیده‌اند؛ بنابراین آن pipeline را بدون بررسی برای ResNet تکرار نمی‌کنیم. دستور `resnet_weights.transforms()` نسخهٔ ارزیابیِ سازگار با همین وزن‌ها را می‌دهد: ضلع کوتاه تصویر به ۲۵۶ می‌رسد، مرکز آن به `224×224` بریده می‌شود، پیکسل‌ها tensor می‌شوند و هر کانال با آمار ImageNet نرمال می‌شود.

سلول زیر یک **عکس از بخش آموزش** را قبل و بعد از `Resize → CenterCrop` نشان می‌دهد و شکل tensor نهایی را چاپ می‌کند. عکس نمایش‌داده‌شده دادهٔ خصوصی است و خروجی تصویری نوت‌بوک وارد Git نمی‌شود. هنوز مدل را آموزش یا ارزیابی نمی‌کنیم. به این دقت کن که بریدن مرکز تصویر ممکن است در عکس‌های باریک بخشی از خودرو را حذف کند؛ بعد از دیدن نمونه دربارهٔ روش مناسب دادهٔ خودمان تصمیم می‌گیریم.


In [ ]:
from PIL import Image
from IPython.display import display
from torchvision import transforms
from torchvision.transforms import InterpolationMode

resnet_eval_transform = resnet_weights.transforms()

# Choose a tall training image so any center-crop loss is easy to see.
sample_image_path = None
for relative_path in resnet_manifest["training_paths"]:
    candidate = project_root / relative_path
    with Image.open(candidate) as image:
        width, height = image.size
    if max(width, height) / min(width, height) >= 1.8:
        sample_image_path = candidate
        break
if sample_image_path is None:
    sample_image_path = project_root / resnet_manifest["training_paths"][0]

with Image.open(sample_image_path) as image:
    original_image = image.convert("RGB")

# For display only: the geometric part before ToTensor and Normalize.
geometry_only = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
])
preview_image = geometry_only(original_image)
resnet_input = resnet_eval_transform(original_image)

print("Original size (width, height):", original_image.size)
print("After resize and center crop:", preview_image.size)
print("Tensor shape (channels, height, width):", tuple(resnet_input.shape))
print("Tensor value range after ImageNet normalization:",
      round(resnet_input.min().item(), 2), round(resnet_input.max().item(), 2))
print("Original image, then model-visible crop:")
display(original_image, preview_image)


## ResNet18، قدم ۴: transform سادهٔ درس استاد

برای مسیر اصلی، آزمایش جانبیِ padding را فعلاً کنار می‌گذاریم. در کد آموزشی استاد، تصویر به‌صورت مستقیم به `224×224` تغییر اندازه داده می‌شود، به tensor تبدیل می‌شود و با میانگین و انحراف معیار ImageNet نرمال می‌شود. این روش تمام تصویر را نگه می‌دارد و center crop ندارد؛ در عکس‌های کشیده ممکن است نسبت طول‌به‌عرض را تغییر دهد. فعلاً همین روش ساده را برای آموزش و validation استفاده می‌کنیم تا feature extraction و سپس fine-tuning را یاد بگیریم. augmentation و مقایسهٔ روش‌های دیگر را بعداً به‌صورت آزمایش جدا انجام می‌دهیم.

سلول بعد فقط transform را تعریف می‌کند و روی همان یک عکس آموزشی شکل tensor را بررسی می‌کند. وزن‌های مدل تغییر نمی‌کنند و هنوز DataLoader یا حلقهٔ آموزش نداریم.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

transformed_sample = resnet_transform(original_image)
print("Original size (width, height):", original_image.size)
print("ResNet tensor shape (channels, height, width):", tuple(transformed_sample.shape))
print("Same normalization as pretrained weights:",
      IMAGENET_MEAN == list(resnet_eval_transform.mean)
      and IMAGENET_STD == list(resnet_eval_transform.std))


## ResNet18، قدم ۵: همان تقسیم واحد را بخوانیم

CNN و ResNet هر دو فهرست آموزش و validation را از یک manifest خصوصی می‌خوانند. این فهرست از ابتدا شامل دو دیتاست، نمونه‌های مجاز `unclean` و نیسان با برچسب هدف `vanet` است. برای سنجش جداگانهٔ نیسان، فقط ردیف‌های نیسانِ **داخل validation** را انتخاب می‌کنیم؛ آن‌ها دوباره به مجموعهٔ آموزش اضافه نمی‌شوند. `fingerprint` کمک می‌کند نتیجه و checkpoint را به تقسیم درست نسبت دهیم.


In [ ]:
import sys
from collections import Counter

sys.path.insert(0, str(project_root))
from scripts.run_experiments import manifest_data

resnet_manifest, resnet_training_rows, resnet_validation_rows = manifest_data()
resnet_neysan_validation_rows = [
    row for row in resnet_validation_rows if row["original_label"] == "neysan"
]
assert resnet_training_rows and resnet_validation_rows and resnet_neysan_validation_rows
print("Split fingerprint:", resnet_manifest["fingerprint"][:12])
print("Training:", len(resnet_training_rows))
print("Validation:", len(resnet_validation_rows))
print("Validation sources:", dict(Counter(row["source"] for row in resnet_validation_rows)))
print("Nissan in validation:", len(resnet_neysan_validation_rows))


## ResNet18، قدم ۶: Dataset و DataLoader

`VehicleDataset` هر مسیر و برچسب را هنگام درخواست به تصویر و اندیس کلاس تبدیل می‌کند. `DataLoader` تصاویر را در batchهای ۳۲تایی می‌خواند. برای آموزش `shuffle=True` و برای validation `shuffle=False` است. `resnet_neysan_loader` تنها نمایی از نیسان‌های داخل validation است؛ تعداد آن را با تعداد batchها اشتباه نگیر. این سلول فقط یک batch آزمایشی می‌خواند و وزن‌ها را تغییر نمی‌دهد.


In [ ]:
import torch
from torch.utils.data import DataLoader
from scripts.run_experiments import VehicleDataset

resnet_train_dataset = VehicleDataset(
    resnet_training_rows, resnet_transform, resnet_manifest["class_to_idx"]
)
resnet_val_dataset = VehicleDataset(
    resnet_validation_rows, resnet_transform, resnet_manifest["class_to_idx"]
)
resnet_neysan_dataset = VehicleDataset(
    resnet_neysan_validation_rows, resnet_transform, resnet_manifest["class_to_idx"]
)
resnet_train_loader = DataLoader(
    resnet_train_dataset, batch_size=32, shuffle=True,
    generator=torch.Generator().manual_seed(42), num_workers=0,
)
resnet_val_loader = DataLoader(
    resnet_val_dataset, batch_size=32, shuffle=False, num_workers=0,
)
resnet_neysan_loader = DataLoader(
    resnet_neysan_dataset, batch_size=32, shuffle=False, num_workers=0,
)
batch_images, batch_labels = next(iter(resnet_train_loader))
print("Dataset lengths:", len(resnet_train_dataset), len(resnet_val_dataset), len(resnet_neysan_dataset))
print("Batches per epoch:", len(resnet_train_loader))
print("Images shape [batch, RGB, height, width]:", tuple(batch_images.shape))
print("Labels shape [batch]:", tuple(batch_labels.shape))
print("First five label indices:", batch_labels[:5].tolist())


## ResNet18، قدم ۷: یک forward pass و محاسبهٔ loss

ابتدا مدل و یک batch را روی یک device می‌گذاریم. برای آنکه آمار داخلی BatchNormِ backbone ثابت بماند، کل مدل را در حالت `eval()` می‌گذاریم و سپس head را با `fc.train()` در حالت آموزش می‌گذاریم؛ این حالت‌ها **فریزِ پارامتر نیستند**. پارامترهای backbone قبلاً با `requires_grad_(False)` فریز شدند و سرِ تازه قابل‌آموزش است. از `torch.no_grad()` استفاده نمی‌کنیم، چون در قدم بعد می‌خواهیم گرادیانِ سر را با `backward()` حساب کنیم.

خروجی `feature_model(images)` باید برای ۳۲ عکس، شکل `[32, 8]` داشته باشد: در هر ردیف هشت **logit/score خام**، نه هشت احتمال. `CrossEntropyLoss` این scoreها و شمارهٔ کلاس درست را می‌گیرد؛ نباید پیش از آن خودمان `softmax` بزنیم. این سلول فقط forward و loss را حساب می‌کند، هیچ وزنی را تغییر نمی‌دهد.


In [ ]:
from scripts.run_experiments import device_for_run

resnet_device = device_for_run()
feature_model = feature_model.to(resnet_device)
feature_model.eval()       # Keep frozen backbone BatchNorm statistics fixed.
feature_model.fc.train()   # The new classifier is the training part.

images_for_resnet = batch_images.to(resnet_device)
labels_for_resnet = batch_labels.to(resnet_device)
class_scores = feature_model(images_for_resnet)
loss_fn = nn.CrossEntropyLoss()
batch_loss = loss_fn(class_scores, labels_for_resnet)

assert class_scores.shape == (len(batch_labels), len(resnet_manifest["class_to_idx"]))
print("Device:", resnet_device)
print("Scores shape [batch, classes]:", tuple(class_scores.shape))
print("One image's eight raw scores:",
      [round(value, 3) for value in class_scores[0].detach().cpu().tolist()])
print("That image's true label index:", labels_for_resnet[0].item())
print("Batch loss:", round(batch_loss.item(), 4))
print("Scores require gradient:", class_scores.requires_grad)
print("Backbone training mode:", feature_model.layer4.training)
print("Classifier training mode:", feature_model.fc.training)


## ResNet18، قدم ۸: گرادیانِ سرِ تازه را بررسی کنیم

`backward()` مشتق loss را برای پارامترهای قابل‌آموزش حساب می‌کند؛ **به‌تنهایی وزن‌ها را تغییر نمی‌دهد**. ابتدا `zero_grad(set_to_none=True)` گرادیان باقی‌مانده از اجرای قبلی سلول را پاک می‌کند. سپس forward و loss را **دوباره** می‌سازیم تا اگر این سلول را تکرار کردی، graph تازه‌ای برای `backward()` داشته باشیم. در پایان باید `.grad` در `conv1` و `layer4` برابر `None` باشد، ولی در `fc.weight` و `fc.bias` وجود داشته باشد. شکل گرادیان `fc.weight` باید `[8, 512]` باشد: به ازای هر کلاس، مشتقِ ۵۱۲ وزن.

این سلول هنوز optimizer و `optimizer.step()` ندارد؛ پس حتی با وجود گرادیانِ `fc`، وزن‌ها همان مقادیر قبلی می‌مانند. این دقیقاً تفاوتِ «محاسبهٔ جهت تغییر» و «اعمال تغییر» است.


In [ ]:
feature_model.eval()
feature_model.fc.train()
feature_model.zero_grad(set_to_none=True)

head_weights_before = feature_model.fc.weight.detach().clone()
fresh_scores = feature_model(images_for_resnet)
fresh_loss = loss_fn(fresh_scores, labels_for_resnet)
fresh_loss.backward()

assert feature_model.conv1.weight.grad is None
assert feature_model.layer4[0].conv1.weight.grad is None
assert feature_model.fc.weight.grad is not None
assert feature_model.fc.bias.grad is not None

print("Conv1 requires_grad:", feature_model.conv1.weight.requires_grad)
print("Conv1 gradient:", feature_model.conv1.weight.grad)
print("Layer4 gradient:", feature_model.layer4[0].conv1.weight.grad)
print("FC requires_grad:", feature_model.fc.weight.requires_grad)
print("FC weight gradient shape:", tuple(feature_model.fc.weight.grad.shape))
print("FC bias gradient shape:", tuple(feature_model.fc.bias.grad.shape))
print("FC gradient norm:", round(feature_model.fc.weight.grad.norm().item(), 4))
print("FC weights unchanged after backward:",
      torch.equal(head_weights_before, feature_model.fc.weight.detach()))


## ResNet18، قدم ۹: یک به‌روزرسانی واقعی برای `fc`

در قدم قبل `backward()` فقط گرادیان را محاسبه کرد و وزن‌ها دست‌نخورده ماندند. این بار یک optimizer از نوع Adam می‌سازیم که **فقط** پارامترهای `feature_model.fc` را دریافت می‌کند. بعد از پاک‌کردن گرادیان‌های قبلی، برای همان batch یک forward و backward تازه انجام می‌دهیم و با `optimizer.step()` وزن‌های head را یک‌بار تغییر می‌دهیم. برای مقایسه، loss همان batch را پس از تغییر دوباره محاسبه می‌کنیم؛ این یک آزمایش آموزشی است و معیار عملکرد روی validation نیست.

این سلول را یک‌بار اجرا کن: هر اجرای دوباره، یک گام آموزشی دیگر روی همان batch خواهد بود.


In [ ]:
head_optimizer = torch.optim.Adam(feature_model.fc.parameters(), lr=1e-3)
feature_model.eval()
feature_model.fc.train()
head_optimizer.zero_grad(set_to_none=True)

head_weights_before_step = feature_model.fc.weight.detach().clone()
scores_before_step = feature_model(images_for_resnet)
loss_before_step = loss_fn(scores_before_step, labels_for_resnet)
loss_before_step.backward()
head_optimizer.step()

with torch.no_grad():
    scores_after_step = feature_model(images_for_resnet)
    loss_after_step = loss_fn(scores_after_step, labels_for_resnet)

weight_change = (feature_model.fc.weight.detach() - head_weights_before_step).norm().item()
assert weight_change > 0
assert feature_model.conv1.weight.grad is None

print("Loss before step:", round(loss_before_step.item(), 4))
print("Loss after step on the same batch:", round(loss_after_step.item(), 4))
print("FC weight change (L2 norm):", round(weight_change, 6))
print("Backbone conv1 still has no gradient:", feature_model.conv1.weight.grad is None)


## ResNet18، قدم ۱۰: یک epoch فقط برای سرِ طبقه‌بندی

تابع قبلیِ `train_one_epoch` برای CNN از `model.train()` استفاده می‌کرد. این بار می‌خواهیم backbone پیش‌آموزش‌دیده و آمار BatchNorm آن ثابت بمانند؛ برای همین ابتدای تابع `model.eval()` و سپس `model.fc.train()` را صدا می‌زنیم. فریز وزن‌ها همچنان به `requires_grad_(False)` مربوط است، نه به `eval()`.

بقیهٔ حلقه همان الگویی است که در CNN خواندیم: هر batch را به device می‌بریم، گرادیان قبلی را پاک می‌کنیم، forward و loss را حساب می‌کنیم، `backward()` و `optimizer.step()` را اجرا می‌کنیم، سپس loss و تعداد پیش‌بینی‌های درست را برای کل epoch جمع می‌کنیم. این سلول فقط تابع را تعریف می‌کند؛ با اجرای آن هنوز epoch آغاز نمی‌شود.


In [ ]:
def train_feature_head_one_epoch(model, loader, optimizer, loss_fn, device):
    model.eval()       # Keep pretrained backbone BatchNorm statistics fixed.
    model.fc.train()   # Train the new classification head.

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        scores = model(images)
        loss = loss_fn(scores, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        correct_predictions += (scores.argmax(dim=1) == labels).sum().item()
        total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }


## ResNet18، قدم ۱۱: مدل تازه و اولین epoch

مدلِ قدم‌های ۷ تا ۹ برای آزمایش یک batch بود و `fc` آن یک‌بار تغییر کرد. برای آزمایش آموزشی یک مدل تازه با وزن‌های پیش‌آموزش‌دیده می‌سازیم، backbone را فریز می‌کنیم، و یک سرِ ۸کلاسهٔ تازه می‌گذاریم. optimizer فقط پارامترهای همین سر را می‌گیرد. سپس **یک epoch کامل** روی تصویرهای مجموعهٔ آموزش فعلی اجرا می‌کنیم. ارزیابی روی تصویرهای validation فعلی در سلول بعدی و جدا از آموزش انجام می‌شود.

این سلول زمان‌برتر از سلول‌های قبلی است: batchهای آموزشی را یک‌بار پردازش می‌کند. اگر دوباره اجرا شود، مدل و optimizer از نو ساخته می‌شوند و همان آزمایش یک‌epochی تکرار می‌شود.


In [ ]:
torch.manual_seed(42)
feature_train_model = models.resnet18(weights=resnet_weights)
feature_train_model.requires_grad_(False)
feature_train_model.fc = nn.Linear(
    feature_train_model.fc.in_features,
    len(resnet_manifest["class_to_idx"]),
)
feature_train_model = feature_train_model.to(resnet_device)
feature_optimizer = torch.optim.Adam(feature_train_model.fc.parameters(), lr=1e-3)

feature_train_metrics = train_feature_head_one_epoch(
    feature_train_model, resnet_train_loader, feature_optimizer, loss_fn, resnet_device
)
print("Train loss:", round(feature_train_metrics["loss"], 4))
print("Train accuracy:", f'{feature_train_metrics["accuracy"]:.1%}')


## ResNet18، قدم ۱۲: ارزیابی جداگانه روی validation

در این سلول مدلِ آموزش‌دیدهٔ قدم قبل را با `eval()` در حالت ارزیابی قرار می‌دهیم. `torch.no_grad()` ساخت گراف گرادیان را برای validation غیرفعال می‌کند؛ هیچ `backward()` یا `optimizer.step()` نداریم، پس وزن‌ها تغییر نمی‌کنند. برای هر batch، loss را در تعداد تصاویر آن ضرب می‌کنیم و در پایان بر کل تصاویر تقسیم می‌کنیم؛ accuracy هم تعداد پیش‌بینی‌های درست تقسیم بر کل تصاویر است.

ارزیابی را در سلول جدا قرار داده‌ایم تا بتوانیم نتیجهٔ مدلِ فعلی را بدون اجرای دوبارهٔ epoch آموزشی ببینیم.


In [ ]:
feature_train_model.eval()
val_loss_sum = 0.0
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in resnet_val_loader:
        images = images.to(resnet_device)
        labels = labels.to(resnet_device)
        scores = feature_train_model(images)
        loss = loss_fn(scores, labels)

        batch_size = images.size(0)
        val_loss_sum += loss.item() * batch_size
        val_correct += (scores.argmax(dim=1) == labels).sum().item()
        val_total += batch_size

feature_val_metrics = {
    "loss": val_loss_sum / val_total,
    "accuracy": val_correct / val_total,
}
print("Train loss:", round(feature_train_metrics["loss"], 4))
print("Train accuracy:", f'{feature_train_metrics["accuracy"]:.1%}')
print("Validation loss:", round(feature_val_metrics["loss"], 4))
print("Validation accuracy:", f'{feature_val_metrics["accuracy"]:.1%}')


## ResNet18، قدم ۱۳: تابع ارزیابیِ قابل‌استفاده در هر epoch

محاسبهٔ validation در قدم قبل را به یک تابع تبدیل می‌کنیم تا بعد از هر epoch دقیقاً همان روش را به کار ببریم. `model.eval()` رفتار BatchNorm را ثابت می‌گذارد و `torch.no_grad()` ساخت گراف گرادیان را غیرفعال می‌کند. این تابع optimizer ندارد و فقط loss و accuracy را برمی‌گرداند؛ با اجرای این سلول هنوز آموزش تازه‌ای آغاز نمی‌شود.


In [ ]:
def evaluate_feature_head(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            scores = model(images)
            loss = loss_fn(scores, labels)

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            correct_predictions += (scores.argmax(dim=1) == labels).sum().item()
            total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }


## ResNet18، قدم ۱۴: ادامهٔ آموزش سرِ طبقه‌بندی تا تعداد epoch انتخاب‌شده

نتیجهٔ epoch اول از `feature_train_metrics` و `feature_val_metrics` وارد `feature_history` می‌شود. پس حلقه از **epoch دوم** شروع می‌کند و مدل و optimizer موجود را ادامه می‌دهد؛ هیچ‌کدام در این سلول از نو ساخته نمی‌شوند. پس از هر epoch، train و validation را جداگانه محاسبه و ثبت می‌کنیم.

طبق الگوی فایل استاد، هر وقت **validation loss کمتر** شد، با `deepcopy(model.state_dict())` یک کپی مستقل از همهٔ وزن‌ها و بافرهای مدل نگه می‌داریم. در پایان، `load_state_dict` بهترین وضعیت را برمی‌گرداند. ممکن است epoch دارای کمترین validation loss با epoch دارای بیشترین validation accuracy یکسان نباشد؛ معیار انتخاب ما در این مرحله loss است. در اجرای ثبت‌شده، `feature_epochs=30` است و این سلول بعد از epoch اول، ۲۹ epoch دیگر اجرا می‌کند؛ بنابراین زمان‌بر است. برای تکرار آزمایش از ابتدا، سلول ساخت مدل و epoch اول را هم دوباره اجرا کن.


In [ ]:
from copy import deepcopy

feature_epochs = 30
feature_history = [{
    "epoch": 1,
    "train_loss": feature_train_metrics["loss"],
    "train_accuracy": feature_train_metrics["accuracy"],
    "val_loss": feature_val_metrics["loss"],
    "val_accuracy": feature_val_metrics["accuracy"],
}]
best_feature_val_loss = feature_val_metrics["loss"]
best_feature_epoch = 1
best_feature_state = deepcopy(feature_train_model.state_dict())

print(
    f"Epoch 01/{feature_epochs} | "
    f"train loss: {feature_train_metrics['loss']:.3f} | "
    f"val loss: {feature_val_metrics['loss']:.3f} | "
    f"train acc: {feature_train_metrics['accuracy']:.1%} | "
    f"val acc: {feature_val_metrics['accuracy']:.1%}"
)

for epoch in range(2, feature_epochs + 1):
    train_metrics = train_feature_head_one_epoch(
        feature_train_model, resnet_train_loader, feature_optimizer, loss_fn, resnet_device
    )
    val_metrics = evaluate_feature_head(
        feature_train_model, resnet_val_loader, loss_fn, resnet_device
    )
    feature_history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
    })

    if val_metrics["loss"] < best_feature_val_loss:
        best_feature_val_loss = val_metrics["loss"]
        best_feature_epoch = epoch
        best_feature_state = deepcopy(feature_train_model.state_dict())

    print(
        f"Epoch {epoch:02d}/{feature_epochs} | "
        f"train loss: {train_metrics['loss']:.3f} | "
        f"val loss: {val_metrics['loss']:.3f} | "
        f"train acc: {train_metrics['accuracy']:.1%} | "
        f"val acc: {val_metrics['accuracy']:.1%}"
    )

feature_train_model.load_state_dict(best_feature_state)
print("Best epoch by validation loss:", best_feature_epoch)
print("Best validation loss:", round(best_feature_val_loss, 4))


## ثبت و مقایسهٔ نتایج

نتایج قبلی در `reports/experiment_results.json` با وضعیت `historical_split` مشخص شده‌اند. فقط نتیجه‌هایی را با هم مقایسه کن که `split_fingerprint` یکسان دارند. بعد از اجرای تازهٔ CNN و ResNet روی تقسیم واحد، معیارهای جدید را در همین فایل ثبت می‌کنیم؛ اعداد قبلی را به تقسیم جدید نسبت نمی‌دهیم. دقت validation داخلی، نتیجهٔ تست مخفی منتورها نیست.


## ResNet18، قدم ۱۵: ذخیرهٔ بهترین وضعیت بر اساس validation loss

`best_feature_state` در حافظهٔ کرنل است. این سلول وزن‌های منتخب، پیش‌پردازش، ترتیب کلاس‌ها و شناسهٔ تقسیم داده را در یک checkpoint محلی ذخیره می‌کند. checkpoint داخل `checkpoints/` می‌ماند و به Git نمی‌رود. epoch و دقت منتخب را از خروجی **اجرای تازه** بخوان؛ عددهای اجرای قبلی به تقسیم دیگری تعلق داشتند.


In [ ]:
checkpoint_dir = project_root / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
feature_checkpoint_path = checkpoint_dir / "resnet18_feature_best_loss.pt"

checkpoint = {
    "architecture": "resnet18",
    "model_state_dict": {
        name: tensor.detach().cpu().clone()
        for name, tensor in feature_train_model.state_dict().items()
    },
    "class_to_idx": resnet_manifest["class_to_idx"],
    "split_fingerprint": resnet_manifest["fingerprint"],
    "preprocessing": {
        "resize": [224, 224],
        "mean": IMAGENET_MEAN,
        "std": IMAGENET_STD,
    },
    "training_mode": "fc_only",
    "selected_epoch": best_feature_epoch,
    "selection_metric": "validation_loss",
    "validation_loss": best_feature_val_loss,
    "validation_accuracy": feature_history[best_feature_epoch - 1]["val_accuracy"],
}
torch.save(checkpoint, feature_checkpoint_path)

print("Saved checkpoint:", feature_checkpoint_path)
print("Selected epoch:", checkpoint["selected_epoch"])
print("Validation loss:", round(checkpoint["validation_loss"], 4))
print("Validation accuracy:", f'{checkpoint["validation_accuracy"]:.1%}')

# Record only the metrics measured on this exact split.
feature_result = {
    "experiment": "resnet18_feature_extraction",
    "architecture": "resnet18", "training_mode": "fc_only",
    "split_fingerprint": resnet_manifest["fingerprint"],
    "result_status": "current_split",
    "training_images": len(resnet_training_rows),
    "validation_images": len(resnet_validation_rows),
    "epochs": len(feature_history),
    "selected_epoch": best_feature_epoch,
    "selection_metric": "validation_loss",
    "selected_validation_loss": best_feature_val_loss,
    "selected_validation_accuracy": feature_history[best_feature_epoch - 1]["val_accuracy"],
    "history": feature_history,
}
results_path = project_root / "reports/experiment_results.json"
all_results = json.loads(results_path.read_text()) if results_path.exists() else {}
all_results["_split"] = {
    "current_fingerprint": resnet_manifest["fingerprint"],
    "comparison_rule": "Compare results only when split_fingerprint matches.",
}
all_results["resnet18_feature_extraction"] = feature_result
results_path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Recorded current feature-extraction result in:", results_path.name)



## ResNet18، قدم ۱۶: آماده‌سازی fine-tuning برای `layer4` و `fc`

در مرحلهٔ قبل، فقط `fc` آموزش دید. اکنون وزن‌های مدل منتخب را از checkpoint قدم ۱۵ می‌خوانیم و از آن یک **کپی مستقل** می‌سازیم تا مدل قبلی برای مقایسه دست‌نخورده بماند. اگر هنوز سلول checkpoint قدم ۱۵ را اجرا نکرده‌ای، پیش از ادامه آن را اجرا کن تا وزن‌های فعلی روی دیسک هم محفوظ باشند.

روی کپی، `layer4` و `fc` را قابل‌آموزش می‌کنیم. لایه‌های `conv1` تا `layer3` همچنان فریز می‌مانند. طبق الگوی فایل استاد، پارامترهای تمام `BatchNorm2d`ها را دوباره فریز می‌کنیم؛ چون `layer4.requires_grad_(True)` در ابتدا BatchNormهای داخل خودِ `layer4` را هم فعال می‌کند. `eval()` روی BatchNorm، رفتار آمار داخلی آن را تنظیم می‌کند و جایگزینِ فریز کردن پارامترهایش نیست. در سلول بعدی باید این حالت را هنگام هر epoch نیز حفظ کنیم.

این سلول فقط انتخاب و شمارش پارامترهاست؛ optimizer و آموزش هنوز شروع نمی‌شوند.


In [ ]:
from copy import deepcopy

# Reload the same selected feature-extraction weights for every new run.
feature_checkpoint = torch.load(feature_checkpoint_path, map_location="cpu", weights_only=True)
if feature_checkpoint["split_fingerprint"] != resnet_manifest["fingerprint"]:
    raise RuntimeError("Feature checkpoint belongs to a different data split.")
if feature_checkpoint["class_to_idx"] != resnet_manifest["class_to_idx"]:
    raise RuntimeError("Feature checkpoint has a different class order.")
fine_tune_model = deepcopy(feature_train_model)
fine_tune_model.load_state_dict(feature_checkpoint["model_state_dict"])
fine_tune_model.layer4.requires_grad_(True)
fine_tune_model.fc.requires_grad_(True)

for module in fine_tune_model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.requires_grad_(False)
        module.eval()

fine_tune_groups = {
    "conv1": fine_tune_model.conv1,
    "layer1": fine_tune_model.layer1,
    "layer2": fine_tune_model.layer2,
    "layer3": fine_tune_model.layer3,
    "layer4": fine_tune_model.layer4,
    "fc": fine_tune_model.fc,
}
for group_name, module in fine_tune_groups.items():
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    total = sum(p.numel() for p in module.parameters())
    print(f"{group_name:7s} | trainable: {trainable:>10,} / total: {total:>10,}")

trainable_names = [
    name for name, parameter in fine_tune_model.named_parameters()
    if parameter.requires_grad
]
assert trainable_names
assert all(name.startswith(("layer4.", "fc.")) for name in trainable_names)
assert all(
    not parameter.requires_grad
    for module in fine_tune_model.modules()
    if isinstance(module, nn.BatchNorm2d)
    for parameter in module.parameters(recurse=False)
)
print("Total trainable parameters:", sum(p.numel() for p in fine_tune_model.parameters() if p.requires_grad))
print("BatchNorm parameters trainable:", 0)


## ResNet18، قدم ۱۷: optimizer با دو نرخ یادگیری

در قدم قبل فقط `requires_grad` پارامترها را تنظیم کردیم. optimizer قدیمیِ `feature_optimizer` فقط `fc` را می‌شناخت؛ باز کردن `layer4` آن را خودکار به optimizer اضافه نمی‌کند. پس یک optimizer تازه با دو گروه می‌سازیم: کانولوشن‌های پیش‌آموزش‌دیدهٔ `layer4` با نرخ یادگیری کوچک‌تر `1e-4` و سرِ طبقه‌بندی `fc` با نرخ `1e-3`. BatchNormها به‌خاطر شرط `requires_grad` وارد گروه `layer4` نمی‌شوند. `weight_decay=1e-4` هم مطابق نمونهٔ استاد به optimizer داده شده است.

پایان سلول بررسی می‌کند هیچ پارامتری در هر دو گروه نباشد، هیچ پارامتر قابل‌آموزشی جا نمانده باشد، و هیچ پارامتری دوبار وارد optimizer نشده باشد. ساخت optimizer **به‌تنهایی وزن‌ها را تغییر نمی‌دهد**؛ هنوز `backward()` و `step()` اجرا نکرده‌ایم.


In [ ]:
LAYER4_LR = 1e-4
HEAD_LR = 1e-3

layer4_parameters = [
    parameter
    for parameter in fine_tune_model.layer4.parameters()
    if parameter.requires_grad
]
head_parameters = [
    parameter
    for parameter in fine_tune_model.fc.parameters()
    if parameter.requires_grad
]

fine_tune_optimizer = torch.optim.Adam(
    [
        {"params": layer4_parameters, "lr": LAYER4_LR, "name": "layer4"},
        {"params": head_parameters, "lr": HEAD_LR, "name": "fc"},
    ],
    weight_decay=1e-4,
)

optimizer_parameters = [
    parameter
    for group in fine_tune_optimizer.param_groups
    for parameter in group["params"]
]
trainable_parameter_ids = {
    id(parameter)
    for parameter in fine_tune_model.parameters()
    if parameter.requires_grad
}
assert set(map(id, layer4_parameters)).isdisjoint(map(id, head_parameters))
assert len(optimizer_parameters) == len({id(parameter) for parameter in optimizer_parameters})
assert {id(parameter) for parameter in optimizer_parameters} == trainable_parameter_ids

for group in fine_tune_optimizer.param_groups:
    count = sum(parameter.numel() for parameter in group["params"])
    print(f"{group['name']:6s} | lr={group['lr']:.0e} | parameters={count:,}")
print("All trainable parameters are in the optimizer exactly once.")


## ResNet18، قدم ۱۸: یک batch آزمایشی پیش از fine-tuning کامل

برای آموزش، کل مدل را با `train()` در حالت آموزش می‌گذاریم؛ سپس همهٔ `BatchNorm2d`ها را دوباره به `eval()` برمی‌گردانیم تا آمار ذخیره‌شدهٔ آن‌ها تغییر نکند. `eval()` روی BatchNorm مسیر گرادیان را برای کانولوشن‌های `layer4` قطع نمی‌کند. با `zero_grad` گرادیان‌های قبلی را پاک می‌کنیم و از یک batch، خروجی، loss و `backward()` می‌گیریم.

باید ببینیم `conv1` گرادیان ندارد، اما convolution داخل `layer4` و وزن‌های `fc` گرادیان دارند. همچنین `running_mean` یک BatchNorm نمونه را قبل و بعد از forward مقایسه می‌کنیم. این سلول **`optimizer.step()` ندارد**؛ پس وزن‌ها و وضعیت Adam هنوز تغییر نمی‌کنند و آزمایش اصلی از همان وزن‌های feature extraction شروع می‌شود.


In [ ]:
fine_tune_model.train()
for module in fine_tune_model.modules():
    if isinstance(module, nn.BatchNorm2d):
        module.eval()

assert all(
    not module.training
    for module in fine_tune_model.modules()
    if isinstance(module, nn.BatchNorm2d)
)

sample_bn = fine_tune_model.layer4[0].bn1
running_mean_before = sample_bn.running_mean.detach().clone()
fine_tune_optimizer.zero_grad(set_to_none=True)

images, labels = next(iter(resnet_train_loader))
images = images.to(resnet_device)
labels = labels.to(resnet_device)
scores = fine_tune_model(images)
loss = loss_fn(scores, labels)
loss.backward()

assert fine_tune_model.conv1.weight.grad is None
assert fine_tune_model.layer4[0].conv1.weight.grad is not None
assert fine_tune_model.fc.weight.grad is not None
assert torch.equal(running_mean_before, sample_bn.running_mean)

print("One-batch loss:", round(loss.item(), 4))
print("Frozen conv1 gradient:", fine_tune_model.conv1.weight.grad)
print("Layer4 gradient norm:", round(fine_tune_model.layer4[0].conv1.weight.grad.norm().item(), 4))
print("FC gradient norm:", round(fine_tune_model.fc.weight.grad.norm().item(), 4))
print("BatchNorm running mean unchanged:", torch.equal(running_mean_before, sample_bn.running_mean))


## ResNet18، قدم ۱۹: تابع یک epoch برای fine-tuning

حلقهٔ آموزش شبیه مرحلهٔ `fc`-only است، اما اکنون `layer4` نیز گرادیان و به‌روزرسانی دارد. ابتدای هر epoch، `model.train()` را صدا می‌زنیم و بلافاصله تمام `BatchNorm2d`ها را دوباره `eval()` می‌کنیم؛ وگرنه آمار ذخیره‌شدهٔ آن‌ها در batchهای آموزش تغییر می‌کند. `requires_grad_(False)` پارامترهای BatchNorm از قدم ۱۶ همچنان برقرار است.

برای هر batch: انتقال به device، پاک‌کردن گرادیان قبلی، forward، loss، `backward()` و `optimizer.step()` انجام می‌شود. در پایان، میانگین loss و دقت کل epoch را برمی‌گردانیم. اجرای این سلول فقط تابع را تعریف می‌کند.


In [ ]:
def train_fine_tune_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        scores = model(images)
        loss = loss_fn(scores, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        correct_predictions += (scores.argmax(dim=1) == labels).sum().item()
        total_images += batch_size

    return {
        "loss": total_loss / total_images,
        "accuracy": correct_predictions / total_images,
    }


## ResNet18، قدم ۲۰: اولین epoch واقعی fine-tuning

درست پیش از آموزش، seed و مولد تصادفی DataLoader را بازنشانی می‌کنیم تا ترتیب batchها در اجراهای تازه یکسان شروع شود. سپس یک epoch کامل روی تصویرهای مجموعهٔ آموزش فعلی اجرا می‌کنیم و با تابع `evaluate_feature_head` روی تصویرهای validation فعلی می‌سنجیم. پیش از شروع، وزنِ یک convolution از `layer4` و آمار یک BatchNorm را کپی می‌کنیم. پس از epoch باید وزن convolution تغییر کرده باشد، ولی `running_mean` آن BatchNorm ثابت مانده باشد. مدلِ فقط-`fc` و checkpoint آن دست‌نخورده می‌مانند.

این سلول همهٔ batchهای آموزشی را پردازش می‌کند و زمان‌بر است. آن را یک‌بار اجرا کن؛ اجرای مجدد یک epoch آموزشی دیگر به همین مدل اضافه می‌کند، ولی برچسب `epoch=1` تاریخچه را از نو می‌نویسد.


In [ ]:
# Reset the training order immediately before epoch 1. The sample batch in
# step 18 and earlier experiments must not shift this run's shuffle sequence.
import random
import numpy as np

if getattr(fine_tune_model, "_fine_tune_started", False):
    raise RuntimeError("This model has already trained. Recreate it from step 16 first.")
FINE_TUNE_SEED = 42
random.seed(FINE_TUNE_SEED)
np.random.seed(FINE_TUNE_SEED)
torch.manual_seed(FINE_TUNE_SEED)
resnet_train_loader.generator.manual_seed(FINE_TUNE_SEED)
fine_tune_model._fine_tune_started = True

layer4_weights_before_epoch = fine_tune_model.layer4[0].conv1.weight.detach().clone()
bn_mean_before_epoch = fine_tune_model.layer4[0].bn1.running_mean.detach().clone()

fine_tune_train_metrics = train_fine_tune_one_epoch(
    fine_tune_model, resnet_train_loader, fine_tune_optimizer, loss_fn, resnet_device
)
fine_tune_val_metrics = evaluate_feature_head(
    fine_tune_model, resnet_val_loader, loss_fn, resnet_device
)
fine_tune_history = [{
    "epoch": 1,
    "train": fine_tune_train_metrics,
    "validation": fine_tune_val_metrics,
}]

layer4_changed = not torch.equal(
    layer4_weights_before_epoch, fine_tune_model.layer4[0].conv1.weight.detach()
)
bn_mean_unchanged = torch.equal(
    bn_mean_before_epoch, fine_tune_model.layer4[0].bn1.running_mean
)
assert layer4_changed
assert bn_mean_unchanged

print("Train loss:", round(fine_tune_train_metrics["loss"], 4))
print("Train accuracy:", f'{fine_tune_train_metrics["accuracy"]:.1%}')
print("Validation loss:", round(fine_tune_val_metrics["loss"], 4))
print("Validation accuracy:", f'{fine_tune_val_metrics["accuracy"]:.1%}')
print("Layer4 convolution weights changed:", layer4_changed)
print("BatchNorm running mean unchanged:", bn_mean_unchanged)


## ResNet18، قدم ۲۱: ادامهٔ fine-tuning تا تعداد epoch انتخاب‌شده

نتیجهٔ epoch اول در `fine_tune_history` وجود دارد. این سلول همان مدل و optimizer را از epoch دوم ادامه می‌دهد؛ بنابراین وزن‌ها از نو مقداردهی نمی‌شوند. پس از هر epoch، train و validation را ثبت و چاپ می‌کنیم. در پایان، بهترین دقت ۱۰ epoch اول و بهترین دقت کل اجرا را جدا چاپ می‌کنیم تا اثر طولانی‌تر کردن آموزش را در **همان مسیر یادگیری** ببینیم.

طبق معیار اصلی فایل استاد، وضعیت دارای **کمترین validation loss** را با `deepcopy(state_dict())` نگه می‌داریم. چون در آزمایش قبلی بیشترین دقت با کمترین loss هم‌زمان نبود، وضعیت دارای **بیشترین validation accuracy** را هم جدا در حافظه نگه می‌داریم. در پایان، `fine_tune_model` را به بهترین وضعیت از نظر loss برمی‌گردانیم. کپیِ بهترین دقت در `best_fine_accuracy_state` باقی می‌ماند تا بعداً بتوانیم آن را جدا ارزیابی یا ذخیره کنیم؛ این سلول هنوز فایل checkpoint جدیدی نمی‌نویسد.

این سلول پس از epoch اول، تا مقدار `FINE_TUNE_EPOCHS` ادامه می‌دهد و زمان‌بر است. آن را فقط یک‌بار در ادامهٔ قدم ۲۰ اجرا کن. مقایسه با مدلِ فقط-`fc` اثر خالصِ بازکردن `layer4` را ثابت نمی‌کند، چون مرحلهٔ fine-tuning گام‌های آموزشی بیشتری هم دارد.

**اجرای دوباره با تعداد epoch متفاوت:** فقط تغییر `FINE_TUNE_EPOCHS` و اجرای دوبارهٔ این سلول، آموزش تازه نیست؛ مدل و optimizer قبلی را ادامه می‌دهد. برای اجرای مستقل ۳۰‌epochی، قدم‌های ۱۶ و ۱۷ را برای ساخت مدل و optimizer جدید، سپس قدم ۲۰ را برای epoch اول، و در پایان همین قدم ۲۱ را اجرا کن. این سلول اگر history از قبل بیش از یک epoch داشته باشد، متوقف می‌شود.


In [ ]:
from copy import deepcopy

# This cell continues the one-epoch run exactly once. Rebuild the model and
# optimizer in steps 16-17, then rerun step 20 before starting a new run.
if len(fine_tune_history) != 1 or fine_tune_history[0]["epoch"] != 1:
    raise RuntimeError(
        "This fine-tuning continuation has already run. "
        "Recreate the fine-tune model and optimizer (steps 16-17), "
        "then rerun the first epoch (step 20) before continuing."
    )

FINE_TUNE_EPOCHS = 30
best_fine_val_loss = fine_tune_val_metrics["loss"]
best_fine_loss_epoch = 1
best_fine_loss_state = deepcopy(fine_tune_model.state_dict())
best_fine_val_accuracy = fine_tune_val_metrics["accuracy"]
best_fine_accuracy_epoch = 1
best_fine_accuracy_state = deepcopy(fine_tune_model.state_dict())

print(
    f"Epoch 01/{FINE_TUNE_EPOCHS} | "
    f"train acc: {fine_tune_train_metrics['accuracy']:.1%} | "
    f"val acc: {fine_tune_val_metrics['accuracy']:.1%} | "
    f"val loss: {fine_tune_val_metrics['loss']:.4f}"
)

for epoch in range(2, FINE_TUNE_EPOCHS + 1):
    train_metrics = train_fine_tune_one_epoch(
        fine_tune_model, resnet_train_loader, fine_tune_optimizer, loss_fn, resnet_device
    )
    val_metrics = evaluate_feature_head(
        fine_tune_model, resnet_val_loader, loss_fn, resnet_device
    )
    fine_tune_history.append({
        "epoch": epoch,
        "train": train_metrics,
        "validation": val_metrics,
    })

    if val_metrics["loss"] < best_fine_val_loss:
        best_fine_val_loss = val_metrics["loss"]
        best_fine_loss_epoch = epoch
        best_fine_loss_state = deepcopy(fine_tune_model.state_dict())
    if val_metrics["accuracy"] > best_fine_val_accuracy:
        best_fine_val_accuracy = val_metrics["accuracy"]
        best_fine_accuracy_epoch = epoch
        best_fine_accuracy_state = deepcopy(fine_tune_model.state_dict())

    print(
        f"Epoch {epoch:02d}/{FINE_TUNE_EPOCHS} | "
        f"train acc: {train_metrics['accuracy']:.1%} | "
        f"val acc: {val_metrics['accuracy']:.1%} | "
        f"val loss: {val_metrics['loss']:.4f}"
    )

# Compare 10 versus 30 epochs along this one training trajectory.
first_ten = fine_tune_history[:min(10, len(fine_tune_history))]
best_first_ten = max(first_ten, key=lambda item: item["validation"]["accuracy"])
print(
    "Best validation accuracy in first 10 epochs:",
    f'{best_first_ten["validation"]["accuracy"]:.1%}',
    "at epoch", best_first_ten["epoch"],
)
print(
    "Best validation accuracy in all epochs:",
    f"{best_fine_val_accuracy:.1%}", "at epoch", best_fine_accuracy_epoch,
)
assert len(fine_tune_history) == FINE_TUNE_EPOCHS
fine_tune_model.load_state_dict(best_fine_loss_state)
print("Best epoch by validation loss:", best_fine_loss_epoch)
print("Best validation loss:", round(best_fine_val_loss, 4))
print("Best epoch by validation accuracy:", best_fine_accuracy_epoch)
print("Best validation accuracy:", f'{best_fine_val_accuracy:.1%}')

# Keep the measured fine-tuning result tied to this split.
fine_result = {
    "experiment": "resnet18_layer4_finetune",
    "architecture": "resnet18", "training_mode": "layer4_plus_fc",
    "split_fingerprint": resnet_manifest["fingerprint"],
    "result_status": "current_split",
    "training_images": len(resnet_training_rows),
    "validation_images": len(resnet_validation_rows),
    "epochs": len(fine_tune_history),
    "seed": FINE_TUNE_SEED,
    "first_10_best_validation_accuracy": best_first_ten["validation"]["accuracy"],
    "first_10_best_validation_accuracy_epoch": best_first_ten["epoch"],
    "selected_epoch": best_fine_loss_epoch,
    "selection_metric": "validation_loss",
    "selected_validation_loss": best_fine_val_loss,
    "selected_validation_accuracy": fine_tune_history[best_fine_loss_epoch - 1]["validation"]["accuracy"],
    "highest_validation_accuracy_epoch": best_fine_accuracy_epoch,
    "highest_validation_accuracy": best_fine_val_accuracy,
    "history": fine_tune_history,
}
results_path = project_root / "reports/experiment_results.json"
all_results = json.loads(results_path.read_text()) if results_path.exists() else {}
all_results["_split"] = {
    "current_fingerprint": resnet_manifest["fingerprint"],
    "comparison_rule": "Compare results only when split_fingerprint matches.",
}
all_results["resnet18_layer4_finetune"] = fine_result
results_path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Recorded current fine-tuning result in:", results_path.name)


## ResNet18، قدم ۲۲: ذخیرهٔ بهترین وزن‌های fine-tuning

در قدم ۲۱ دو نسخهٔ مستقل از وزن‌ها در حافظه نگه داشتیم: `best_fine_loss_state` برای کمترین validation loss و `best_fine_accuracy_state` برای بیشترین validation accuracy. این سلول هر دو را با برچسب معیار انتخاب، epoch، ترتیب کلاس‌ها، پیش‌پردازش و شناسهٔ تقسیم داده در `checkpoints/` ذخیره می‌کند. فایل‌ها خصوصی و خارج از Git می‌مانند. ذخیره‌سازی آموزش تازه‌ای انجام نمی‌دهد؛ پیش از بستن کرنل اجرا کن.


In [ ]:
from pathlib import Path

fine_checkpoint_dir = project_root / "checkpoints"
fine_checkpoint_dir.mkdir(parents=True, exist_ok=True)


def save_fine_tune_checkpoint(path, state, epoch, selection_metric):
    metrics = fine_tune_history[epoch - 1]["validation"]
    payload = {
        "architecture": "resnet18",
        "training_mode": "layer4_plus_fc",
        "model_state_dict": {
            name: tensor.detach().cpu().clone()
            for name, tensor in state.items()
        },
        "class_to_idx": resnet_manifest["class_to_idx"],
        "split_fingerprint": resnet_manifest["fingerprint"],
        "preprocessing": {
            "resize": [224, 224],
            "mean": IMAGENET_MEAN,
            "std": IMAGENET_STD,
        },
        "training_config": {
            "seed": globals().get("FINE_TUNE_SEED"),
            "optimizer": "Adam",
            "layer4_lr": LAYER4_LR,
            "head_lr": HEAD_LR,
            "weight_decay": 1e-4,
            "batch_size": 32,
            "epochs": len(fine_tune_history),
        },
        "selected_epoch": epoch,
        "selection_metric": selection_metric,
        "validation_loss": metrics["loss"],
        "validation_accuracy": metrics["accuracy"],
    }
    torch.save(payload, path)
    saved = torch.load(path, map_location="cpu", weights_only=True)
    assert saved["split_fingerprint"] == resnet_manifest["fingerprint"]
    assert saved["selected_epoch"] == epoch
    return saved


loss_checkpoint = save_fine_tune_checkpoint(
    fine_checkpoint_dir / "resnet18_layer4_best_loss.pt",
    best_fine_loss_state,
    best_fine_loss_epoch,
    "validation_loss",
)
accuracy_checkpoint = save_fine_tune_checkpoint(
    fine_checkpoint_dir / "resnet18_layer4_best_accuracy.pt",
    best_fine_accuracy_state,
    best_fine_accuracy_epoch,
    "validation_accuracy",
)
print("Saved by loss:", loss_checkpoint["selected_epoch"],
      f'{loss_checkpoint["validation_accuracy"]:.1%}')
print("Saved by accuracy:", accuracy_checkpoint["selected_epoch"],
      f'{accuracy_checkpoint["validation_accuracy"]:.1%}')


## ResNet18، قدم ۲۳: بازیابی مدل منتخب و ارزیابی روی validation

مدلی را که در قدم ۲۲ بر اساس کمترین **validation loss** ذخیره کردیم از فایل می‌خوانیم. ساختار ResNet18 را بدون دانلود دوبارهٔ وزن‌های ImageNet می‌سازیم، سپس وزن‌های آموزش‌دیدهٔ خودمان را بارگذاری می‌کنیم. تطابق تقسیم داده و ترتیب کلاس‌ها را بررسی می‌کنیم تا checkpoint اشتباهی ارزیابی نشود. این سلول فقط ارزیابی می‌کند و وزن‌ها را تغییر نمی‌دهد؛ `validation` همان دادهٔ داخلی ماست، نه تست پنهان منتورها.


In [ ]:
selected_checkpoint_path = project_root / "checkpoints" / "resnet18_layer4_best_loss.pt"
selected_checkpoint = torch.load(
    selected_checkpoint_path, map_location="cpu", weights_only=True
)
assert selected_checkpoint["split_fingerprint"] == resnet_manifest["fingerprint"]
assert selected_checkpoint["class_to_idx"] == resnet_manifest["class_to_idx"]

selected_model = models.resnet18(weights=None)
selected_model.fc = nn.Linear(
    selected_model.fc.in_features, len(resnet_manifest["class_to_idx"])
)
selected_model.load_state_dict(selected_checkpoint["model_state_dict"])
selected_model = selected_model.to(resnet_device)

selected_metrics = evaluate_feature_head(
    selected_model, resnet_val_loader, loss_fn, resnet_device
)
print("Selected epoch:", selected_checkpoint["selected_epoch"])
print("Saved validation loss:", round(selected_checkpoint["validation_loss"], 4))
print("Reloaded validation loss:", round(selected_metrics["loss"], 4))
print("Saved validation accuracy:", f'{selected_checkpoint["validation_accuracy"]:.1%}')
print("Reloaded validation accuracy:", f'{selected_metrics["accuracy"]:.1%}')


## ResNet18، قدم ۲۴: پیش‌بینی برای هر تصویر validation

تا اینجا فقط loss و accuracy کلی را داشتیم. برای محاسبهٔ precision، recall و F1 هر کلاس، باید برچسب واقعی و پیش‌بینی مدل را برای **هر تصویر** نگه داریم. مدل در حالت `eval()` قرار می‌گیرد و `torch.no_grad()` جلوی ساخت گراف گرادیان را می‌گیرد، چون اینجا آموزشی انجام نمی‌شود. `softmax` امتیازهای خام مدل را به احتمال‌های هشت کلاس تبدیل می‌کند؛ بیشترین احتمال، اطمینان مدل به کلاس پیش‌بینی‌شده است. `shuffle=False` در DataLoader باعث می‌شود ترتیب این خروجی‌ها با ردیف‌های validation یکسان بماند.


In [ ]:
selected_model.eval()
resnet_y_true = []
resnet_y_pred = []
resnet_confidences = []

with torch.no_grad():
    for images, labels in resnet_val_loader:
        images = images.to(resnet_device)
        scores = selected_model(images)
        probabilities = torch.softmax(scores, dim=1)
        confidences, predictions = probabilities.max(dim=1)

        resnet_y_true.extend(labels.tolist())
        resnet_y_pred.extend(predictions.cpu().tolist())
        resnet_confidences.extend(confidences.cpu().tolist())

assert len(resnet_y_true) == len(resnet_val_dataset)
assert len(resnet_y_pred) == len(resnet_y_true)
resnet_correct = sum(
    true_label == predicted_label
    for true_label, predicted_label in zip(resnet_y_true, resnet_y_pred)
)
print("Validation images:", len(resnet_y_true))
print("Correct predictions:", resnet_correct)
print("Validation accuracy:", f"{resnet_correct / len(resnet_y_true):.1%}")
print("First five true labels:", resnet_y_true[:5])
print("First five predictions:", resnet_y_pred[:5])
print("First five confidences:", [round(value, 3) for value in resnet_confidences[:5]])


## ResNet18، قدم ۲۵: معیارهای جداگانهٔ هر کلاس

دقت کلی به‌تنهایی نمی‌گوید مدل برای کدام خودرو ضعیف‌تر است. `classification_report` برای هر کلاس، `precision`، `recall`، `F1-score` و `support` را نشان می‌دهد. هر کلاس موقتاً در برابر «همهٔ کلاس‌های دیگر» سنجیده می‌شود:

- `precision`: از همهٔ تصویرهایی که مدل **این کلاس** نامیده، چند تا واقعاً همین کلاس‌اند؟
- `recall`: از همهٔ تصویرهای واقعی **این کلاس**، مدل چند تا را پیدا کرده است؟
- `F1`: میانگین هماهنگ precision و recall؛ وقتی یکی پایین باشد، F1 هم پایین می‌آید.
- `support`: تعداد تصویرهای واقعی آن کلاس در validation.

`macro avg` میانگین سادهٔ هشت کلاس است و به هر کلاس وزن یکسان می‌دهد. این گزارش فقط روی validation داخلی است.


In [ ]:
from sklearn.metrics import classification_report

resnet_class_names = [
    name
    for name, index in sorted(
        resnet_manifest["class_to_idx"].items(), key=lambda item: item[1]
    )
]
resnet_class_report = classification_report(
    resnet_y_true,
    resnet_y_pred,
    labels=list(range(len(resnet_class_names))),
    target_names=resnet_class_names,
    digits=3,
    zero_division=0,
)
print(resnet_class_report)


## ResNet18، قدم ۲۶: ماتریس خطای شمارشی و نرمال‌شده

در هر دو نمودار، **سطر = کلاس واقعی** و **ستون = کلاس پیش‌بینی‌شده** است. اعداد روی قطر اصلی پیش‌بینی‌های درست‌اند؛ خانه‌های بیرون قطر خطاها را نشان می‌دهند. نمودار سمت چپ تعداد تصویرها را نشان می‌دهد. در نمودار سمت راست هر سطر بر تعداد تصاویر واقعی همان کلاس تقسیم شده است؛ بنابراین می‌توان کلاس‌هایی با تعداد نمونهٔ متفاوت را بهتر مقایسه کرد. این همان validation داخلی است و آموزش تازه‌ای انجام نمی‌شود.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

class_indices = list(range(len(resnet_class_names)))
resnet_cm_counts = confusion_matrix(
    resnet_y_true, resnet_y_pred, labels=class_indices
)
resnet_cm_normalized = confusion_matrix(
    resnet_y_true, resnet_y_pred, labels=class_indices, normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
ConfusionMatrixDisplay(
    resnet_cm_counts, display_labels=resnet_class_names
).plot(ax=axes[0], cmap="Blues", values_format="d", colorbar=False,
       xticks_rotation=45)
axes[0].set_title("Validation: number of images")

ConfusionMatrixDisplay(
    resnet_cm_normalized, display_labels=resnet_class_names
).plot(ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False,
       xticks_rotation=45)
axes[1].set_title("Validation: fraction of each true class")

plt.tight_layout()
plt.show()


## ResNet18، قدم ۲۷: بررسی همهٔ خطاهای validation

تمام تصویرهایی را که مدل اشتباه پیش‌بینی کرده، به‌ترتیب بیشترین اطمینانِ `softmax` در صفحه‌های ۱۲تایی نشان می‌دهیم. هر صفحه یک شبکهٔ `3×4` است؛ اگر صفحهٔ آخر کمتر از ۱۲ تصویر داشته باشد، خانه‌های خالی پنهان می‌شوند. برچسب واقعی، پیش‌بینی و اطمینان بالای هر تصویر آمده است. این تصویرهای اصلی برای بررسی انسانی‌اند؛ مدل آن‌ها را با transform اندازهٔ `224×224` دریافت کرده است. این سلول آموزش تازه یا crop انجام نمی‌دهد.


In [ ]:
from PIL import Image

resnet_wrong_indices = [
    index
    for index, (true_label, predicted_label) in enumerate(
        zip(resnet_y_true, resnet_y_pred)
    )
    if true_label != predicted_label
]
resnet_top_wrong_indices = sorted(
    resnet_wrong_indices,
    key=lambda index: resnet_confidences[index],
    reverse=True,
)

page_size = 12
for page_start in range(0, len(resnet_top_wrong_indices), page_size):
    page_indices = resnet_top_wrong_indices[page_start:page_start + page_size]
    fig, axes = plt.subplots(3, 4, figsize=(15, 11))
    for axis, index in zip(axes.flat, page_indices):
        row = resnet_val_dataset.rows[index]
        with Image.open(project_root / row["path"]) as image:
            axis.imshow(image.convert("RGB"))
        true_name = resnet_class_names[resnet_y_true[index]]
        predicted_name = resnet_class_names[resnet_y_pred[index]]
        axis.set_title(
            f"True: {true_name} | Pred: {predicted_name}\n"
            f"Confidence: {resnet_confidences[index]:.1%}"
        )
        axis.axis("off")
    for axis in list(axes.flat)[len(page_indices):]:
        axis.axis("off")
    fig.suptitle(
        f"Mistakes {page_start + 1}–{page_start + len(page_indices)} "
        f"of {len(resnet_top_wrong_indices)}",
        y=1.01,
    )
    plt.tight_layout()
    plt.show()

print("Validation mistakes:", len(resnet_wrong_indices))
print("Mistakes actually shown:", len(resnet_top_wrong_indices))


## ResNet18، قدم ۲۸: مقایسهٔ دو checkpoint روی یک validation

در قدم ۲۲ دو نسخه ذخیره کردیم: یکی با کمترین `validation loss` و دیگری با بیشترین `validation accuracy`. هر دو را روی **همان ۸۶۷ تصویر validation فعلی** مقایسه می‌کنیم. علاوه بر loss و accuracy، `macro-F1` را هم می‌بینیم تا عملکرد هر هشت کلاس در تصمیم نهایی دیده شود. این فقط ارزیابی است؛ هیچ وزنی آموزش داده یا تغییر نمی‌شود. نتیجهٔ تست پنهان منتورها هنوز معلوم نیست.


In [ ]:
from sklearn.metrics import f1_score

highest_accuracy_path = project_root / "checkpoints" / "resnet18_layer4_best_accuracy.pt"
highest_accuracy_checkpoint = torch.load(
    highest_accuracy_path, map_location="cpu", weights_only=True
)
assert highest_accuracy_checkpoint["split_fingerprint"] == resnet_manifest["fingerprint"]
assert highest_accuracy_checkpoint["class_to_idx"] == resnet_manifest["class_to_idx"]

highest_accuracy_model = models.resnet18(weights=None)
highest_accuracy_model.fc = nn.Linear(
    highest_accuracy_model.fc.in_features, len(resnet_class_names)
)
highest_accuracy_model.load_state_dict(highest_accuracy_checkpoint["model_state_dict"])
highest_accuracy_model = highest_accuracy_model.to(resnet_device)
highest_accuracy_model.eval()

accuracy_model_true = []
accuracy_model_pred = []
accuracy_model_loss_sum = 0.0
with torch.no_grad():
    for images, labels in resnet_val_loader:
        scores = highest_accuracy_model(images.to(resnet_device))
        batch_loss = loss_fn(scores, labels.to(resnet_device))
        accuracy_model_loss_sum += batch_loss.item() * len(labels)
        accuracy_model_true.extend(labels.tolist())
        accuracy_model_pred.extend(scores.argmax(dim=1).cpu().tolist())

assert accuracy_model_true == resnet_y_true
loss_model_macro_f1 = f1_score(resnet_y_true, resnet_y_pred, average="macro")
accuracy_model_macro_f1 = f1_score(
    accuracy_model_true, accuracy_model_pred, average="macro"
)
print(
    f"Lowest-loss checkpoint (epoch {selected_checkpoint['selected_epoch']}): "
    f"loss={selected_metrics['loss']:.4f}, "
    f"accuracy={selected_metrics['accuracy']:.1%}, macro-F1={loss_model_macro_f1:.3f}"
)
print(
    f"Highest-accuracy checkpoint (epoch {highest_accuracy_checkpoint['selected_epoch']}): "
    f"loss={accuracy_model_loss_sum / len(accuracy_model_true):.4f}, "
    f"accuracy={sum(a == b for a, b in zip(accuracy_model_true, accuracy_model_pred)) / len(accuracy_model_true):.1%}, "
    f"macro-F1={accuracy_model_macro_f1:.3f}"
)


## ResNet18، قدم ۲۹: گزارش هر کلاس برای checkpoint با بیشترین دقت

در قدم قبل، نسخهٔ epoch ۱۹ روی همین validation دقت و `macro-F1` بیشتری داشت. حالا `precision`، `recall` و `F1` تک‌تک کلاس‌ها را برای همین نسخه چاپ می‌کنیم و با گزارش نسخهٔ کمترین loss در قدم ۲۵ مقایسه می‌کنیم. این سلول از پیش‌بینی‌های موجودِ قدم ۲۸ استفاده می‌کند؛ مدل دوباره آموزش یا حتی اجرا نمی‌شود.


In [ ]:
from sklearn.metrics import classification_report

accuracy_model_mistakes = sum(
    true_label != predicted_label
    for true_label, predicted_label in zip(accuracy_model_true, accuracy_model_pred)
)
print("Checkpoint selected by highest validation accuracy")
print("Epoch:", highest_accuracy_checkpoint["selected_epoch"])
print("Validation mistakes:", accuracy_model_mistakes)
print(classification_report(
    accuracy_model_true,
    accuracy_model_pred,
    labels=list(range(len(resnet_class_names))),
    target_names=resnet_class_names,
    digits=3,
    zero_division=0,
))


## ResNet18، قدم ۳۰: بررسی آستانهٔ بازبینی انسانی

در صورت پروژه، پیش‌بینی‌های نامطمئن باید برای بررسی انسانی علامت بخورند. اینجا فقط روی **validation** نسخهٔ منتخبِ epoch ۱۹ کار می‌کنیم. خروجی `softmax` برای هر تصویر هشت عدد می‌دهد که جمعشان یک است؛ بزرگ‌ترین عدد را «confidence» پیش‌بینی می‌نامیم. این عدد الزاماً احتمال واقعیِ درست‌بودن پاسخ نیست.

برای چند آستانه، تعداد تصویرهای ارجاع‌شده، تعداد خطاهای کشف‌شده در آن‌ها و نرخ خطا در تصویرهای پذیرفته‌شده را می‌سنجیم. هنوز آستانهٔ نهایی را انتخاب نمی‌کنیم؛ اول باید خروجی را ببینیم. مدل در این سلول آموزش نمی‌بیند.


In [ ]:
accuracy_model_confidences = []
accuracy_model_predictions_check = []

highest_accuracy_model.eval()
with torch.no_grad():
    for images, _ in resnet_val_loader:
        scores = highest_accuracy_model(images.to(resnet_device))
        probabilities = torch.softmax(scores, dim=1)
        confidence, prediction = probabilities.max(dim=1)
        accuracy_model_confidences.extend(confidence.cpu().tolist())
        accuracy_model_predictions_check.extend(prediction.cpu().tolist())

assert accuracy_model_predictions_check == accuracy_model_pred
assert len(accuracy_model_confidences) == len(accuracy_model_true)

mistake_flags = [
    true_label != predicted_label
    for true_label, predicted_label in zip(accuracy_model_true, accuracy_model_pred)
]
total_mistakes = sum(mistake_flags)
print(f"Validation images: {len(mistake_flags)} | Mistakes: {total_mistakes}")
print("Threshold | Review images | Mistakes sent to review | Error rate among accepted")

for threshold in (0.50, 0.70, 0.90, 0.95):
    review_indices = [
        index for index, confidence in enumerate(accuracy_model_confidences)
        if confidence < threshold
    ]
    accepted_indices = [
        index for index, confidence in enumerate(accuracy_model_confidences)
        if confidence >= threshold
    ]
    reviewed_mistakes = sum(mistake_flags[index] for index in review_indices)
    accepted_mistakes = sum(mistake_flags[index] for index in accepted_indices)
    accepted_error_rate = (
        accepted_mistakes / len(accepted_indices) if accepted_indices else 0.0
    )
    print(
        f"{threshold:>9.2f} | {len(review_indices):>13} "
        f"| {reviewed_mistakes:>23}/{total_mistakes} "
        f"| {accepted_mistakes:>3}/{len(accepted_indices):<3} "
        f"({accepted_error_rate:.1%})"
    )


## ResNet18، قدم ۳۱: ارزیابی جداگانهٔ نیسان در validation

در تقسیم‌بندی فعلی، نیسان به برچسب هشت‌کلاسهٔ `vanet` نگاشت شده است. **۱۹۹ نیسان در training و ۵۰ نیسان در validation داریم**؛ پس این ۵۰ تصویر نمونه‌های اعتبارسنجیِ جدا از آموزش‌اند، ولی نیسان برای مدل یک نوع کاملاً ندیده نیست. `resnet_neysan_validation_rows` زیرمجموعهٔ همان validation است، نه یک مجموعهٔ تست مستقل.

از پیش‌بینی‌ها و confidenceهایی که در قدم‌های ۲۸ و ۳۰ محاسبه شدند استفاده می‌کنیم. نیسان را با سایر وانت‌ها و کل validation مقایسه می‌کنیم. برای آستانهٔ موقت `0.90` می‌سنجیم چند تصویر به بازبینی می‌رود و چند خطا با وجود confidence بالا پذیرفته می‌شود. این سلول وزن‌ها را تغییر نمی‌دهد. **پیش از پایان پروژه، مفهوم confidence و انتخاب آستانه را با جزئیات دوباره مرور می‌کنیم.**


In [ ]:
from IPython.display import HTML, display

assert len(resnet_validation_rows) == len(accuracy_model_true)
assert len(accuracy_model_confidences) == len(accuracy_model_true)
assert all(
    resnet_manifest["class_to_idx"][row["label"]] == true_label
    for row, true_label in zip(resnet_validation_rows, accuracy_model_true)
)

neysan_indices = [
    index for index, row in enumerate(resnet_validation_rows)
    if row["source"] == "neysan"
]
other_vanet_indices = [
    index for index, row in enumerate(resnet_validation_rows)
    if row["label"] == "vanet" and row["source"] != "neysan"
]
assert len(neysan_indices) == len(resnet_neysan_validation_rows)

review_threshold = 0.90  # Candidate from step 30; not a final decision yet.
print("Training Nissan images:", sum(row["source"] == "neysan" for row in resnet_training_rows))
print("Validation Nissan images:", len(neysan_indices))

summary_rows = []
for group_name, indices in (
    ("Nissan", neysan_indices),
    ("Other vanet", other_vanet_indices),
    ("All validation", list(range(len(accuracy_model_true)))),
):
    correct = sum(
        accuracy_model_true[index] == accuracy_model_pred[index]
        for index in indices
    )
    review_count = sum(
        accuracy_model_confidences[index] < review_threshold
        for index in indices
    )
    accepted_errors = sum(
        accuracy_model_true[index] != accuracy_model_pred[index]
        and accuracy_model_confidences[index] >= review_threshold
        for index in indices
    )
    summary_rows.append((
        group_name, len(indices), correct, f"{correct / len(indices):.1%}",
        review_count, accepted_errors,
    ))

headers = ("Group", "Images", "Correct", "Accuracy", "Review (<0.90)", "Errors accepted")
header_html = "".join(f"<th>{heading}</th>" for heading in headers)
rows_html = "".join(
    "<tr>" + "".join(f"<td>{value}</td>" for value in row) + "</tr>"
    for row in summary_rows
)
display(HTML(
    '<table border="1" cellpadding="6" style="border-collapse:collapse;text-align:center">'
    f"<thead><tr>{header_html}</tr></thead><tbody>{rows_html}</tbody></table>"
))


## ResNet18، قدم ۳۲: رابطهٔ خطاهای بین کلاس‌ها

ماتریس قدم ۲۶ مربوط به checkpoint با **کمترین loss** بود. حالا همان دو ماتریس را برای checkpoint منتخب با **بیشترین دقت validation** (epoch ۱۹) می‌سازیم. در ماتریس، سطر کلاس واقعی و ستون پیش‌بینی مدل است. ماتریس شمارشی تعداد تصاویر را نشان می‌دهد؛ ماتریس نرمال‌شده هر سطر را بر تعداد تصاویر همان کلاس تقسیم می‌کند تا کلاس‌های پرجمعیت مقایسه را منحرف نکنند.

برای هر جفت کلاس `i` و `j`، دو جهت خطا را جمع می‌زنیم: `C_normalized[i, j] + C_normalized[j, i]`. پنج جفت با بیشترین خطای دوطرفه چاپ می‌شوند. این امتیاز فقط می‌گوید کدام جفت ارزش بررسی دارد؛ به‌تنهایی دلیل ادغام دو کلاس نیست. هیچ آموزشی در این سلول انجام نمی‌شود.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

class_indices = list(range(len(resnet_class_names)))
accuracy_cm_counts = confusion_matrix(
    accuracy_model_true, accuracy_model_pred, labels=class_indices
)
accuracy_cm_normalized = confusion_matrix(
    accuracy_model_true, accuracy_model_pred,
    labels=class_indices, normalize="true",
)

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
ConfusionMatrixDisplay(
    accuracy_cm_counts, display_labels=resnet_class_names
).plot(ax=axes[0], cmap="Blues", values_format="d", colorbar=False,
       xticks_rotation=45)
axes[0].set_title("Selected checkpoint: image counts")

ConfusionMatrixDisplay(
    accuracy_cm_normalized, display_labels=resnet_class_names
).plot(ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False,
       xticks_rotation=45)
axes[1].set_title("Selected checkpoint: fraction of each true class")
plt.tight_layout()
plt.show()

pair_confusions = []
for first in range(len(resnet_class_names)):
    for second in range(first + 1, len(resnet_class_names)):
        pair_score = (
            accuracy_cm_normalized[first, second]
            + accuracy_cm_normalized[second, first]
        )
        pair_confusions.append((pair_score, first, second))
pair_confusions.sort(key=lambda item: item[0], reverse=True)

print("Five most mutually confused class pairs:")
for rank, (score, first, second) in enumerate(pair_confusions[:5], start=1):
    first_name = resnet_class_names[first]
    second_name = resnet_class_names[second]
    print(
        f"{rank}. {first_name} ↔ {second_name}: score={score:.3f}; "
        f"{first_name} → {second_name}: {accuracy_cm_counts[first, second]}, "
        f"{second_name} → {first_name}: {accuracy_cm_counts[second, first]}"
    )


## ResNet18، قدم ۳۳: دیدن خطاهای کامیون و کامیونت

ماتریس قدم قبل نشان داد ۱۲ کامیون به‌اشتباه «کامیونت» و ۳ کامیونت به‌اشتباه «کامیون» پیش‌بینی شده‌اند. اینجا **همان ۱۵ تصویرِ validation** را به‌ترتیب بیشترین confidence می‌بینیم. عنوان هر تصویر برچسب موجود، پیش‌بینی مدل و confidence را نشان می‌دهد. دیدن نمونه‌ها برای تفسیر خطا لازم است؛ عدد ماتریس به‌تنهایی نمی‌گوید برچسب داده غلط است یا مدل به‌خاطر شباهت ظاهری اشتباه کرده است.

این تصاویر از دادهٔ خصوصی هستند؛ خروجی تصویری نوت‌بوک باید پیش از commit عمومی پاک شود. این سلول فایل داده را جابه‌جا نمی‌کند و مدلی را آموزش نمی‌دهد.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

kamyun_index = resnet_manifest["class_to_idx"]["kamyun"]
kamyunet_index = resnet_manifest["class_to_idx"]["kamyunet"]
pair_error_indices = [
    index
    for index, (true_label, predicted_label) in enumerate(
        zip(accuracy_model_true, accuracy_model_pred)
    )
    if (true_label, predicted_label) in (
        (kamyun_index, kamyunet_index),
        (kamyunet_index, kamyun_index),
    )
]
pair_error_indices.sort(
    key=lambda index: accuracy_model_confidences[index], reverse=True
)
assert len(pair_error_indices) == (
    accuracy_cm_counts[kamyun_index, kamyunet_index]
    + accuracy_cm_counts[kamyunet_index, kamyun_index]
)

print("Truck/trucklet mistakes:", len(pair_error_indices))
fig, axes = plt.subplots(3, 5, figsize=(19, 11))
for rank, (axis, index) in enumerate(zip(axes.flat, pair_error_indices), start=1):
    row = resnet_validation_rows[index]
    with Image.open(project_root / row["path"]) as image:
        axis.imshow(image.convert("RGB"))
    true_name = resnet_class_names[accuracy_model_true[index]]
    predicted_name = resnet_class_names[accuracy_model_pred[index]]
    axis.set_title(
        f"#{rank} true: {true_name} → pred: {predicted_name}\n"
        f"confidence: {accuracy_model_confidences[index]:.2f}",
        fontsize=9,
    )
    axis.axis("off")
    print(f"#{rank:02d}: {row['path']}")
for axis in list(axes.flat)[len(pair_error_indices):]:
    axis.axis("off")
plt.tight_layout()
plt.show()


## ResNet18، قدم ۳۴: تصمیم کلاس‌ها و checkpoint پیش‌بینی

در checkpoint منتخب، ۱۲ کامیون به‌صورت کامیونت و ۳ کامیونت به‌صورت کامیون پیش‌بینی شدند. در گالری، چند تصویرِ دارای برچسب کامیون از این زاویه شبیه کامیونت‌های سبک‌اند؛ اما فقط با ظاهر یک قاب نمی‌توان با اطمینان برچسب درست را تعیین کرد. این موارد برای بازبینی داده مناسب‌اند. **فعلاً کلاس‌های کامیون و کامیونت را ادغام نمی‌کنیم**: تمایز آن‌ها برای مسئله و تست نهایی مهم است، خطا نیز در دو جهت متقارن نیست. بدون برچسب‌های بررسی‌شده، ادغام می‌تواند مسئله را پنهان کند.

برای مدل پیش‌بینی، checkpoint با بیشترین دقت validation (epoch ۱۹) را برمی‌داریم. آستانهٔ `0.90` را از بررسی validation انتخاب می‌کنیم: نسبت به `0.70`، با ۲۰ ارجاع بیشتر، ۷ خطای بیشتر به صف بازبینی می‌رود؛ `0.95` چهار ارجاع بیشتر داشت ولی خطای بیشتری پیدا نکرد. **این آستانه فقط برای دادهٔ مشابه validation توجیه شده است**: هنوز ۲۳ خطا با confidence حداقل `0.90` پذیرفته می‌شوند و آزمون پنهان منتورها را ندیده‌ایم.

سلول بعد یک کپی خصوصی از checkpoint منتخب با metadata آستانه می‌سازد؛ وزن‌ها و آموزش تغییر نمی‌کنند. فایل داخل `checkpoints/` است و طبق `.gitignore` وارد Git نمی‌شود.


In [ ]:
FINAL_REVIEW_THRESHOLD = 0.90
reviewed_indices = [
    index for index, confidence in enumerate(accuracy_model_confidences)
    if confidence < FINAL_REVIEW_THRESHOLD
]
accepted_indices = [
    index for index, confidence in enumerate(accuracy_model_confidences)
    if confidence >= FINAL_REVIEW_THRESHOLD
]
reviewed_errors = sum(mistake_flags[index] for index in reviewed_indices)
accepted_errors = sum(mistake_flags[index] for index in accepted_indices)
assert reviewed_errors + accepted_errors == accuracy_model_mistakes

final_checkpoint = dict(highest_accuracy_checkpoint)
final_checkpoint["review_threshold"] = FINAL_REVIEW_THRESHOLD
final_checkpoint["threshold_source"] = "validation"
final_checkpoint["threshold_validation"] = {
    "images": len(accuracy_model_true),
    "mistakes": accuracy_model_mistakes,
    "sent_to_review": len(reviewed_indices),
    "mistakes_sent_to_review": reviewed_errors,
    "mistakes_accepted": accepted_errors,
}
final_checkpoint_path = project_root / "checkpoints" / "resnet18_final.pt"
torch.save(final_checkpoint, final_checkpoint_path)

reloaded_final_checkpoint = torch.load(
    final_checkpoint_path, map_location="cpu", weights_only=True
)
assert reloaded_final_checkpoint["split_fingerprint"] == resnet_manifest["fingerprint"]
assert reloaded_final_checkpoint["class_to_idx"] == resnet_manifest["class_to_idx"]
assert reloaded_final_checkpoint["review_threshold"] == FINAL_REVIEW_THRESHOLD
print("Selected epoch:", reloaded_final_checkpoint["selected_epoch"])
print("Review threshold:", reloaded_final_checkpoint["review_threshold"])
print("Images sent to review:", len(reviewed_indices))
print("Mistakes sent to review:", reviewed_errors)
print("Mistakes accepted:", accepted_errors)
print("Saved private checkpoint:", final_checkpoint_path)


## ResNet18، قدم ۳۵: آزمایش تابع پیش‌بینی و JSON

فایل `predict.py` رابط استفاده از مدلِ آموزش‌دیده است. تابع `predict_image` ساختار ResNet18 را می‌سازد، وزن‌ها و اطلاعات کلاس‌ها را از checkpoint می‌خواند، همان `Resize → ToTensor → Normalize` زمان validation را انجام می‌دهد و با `eval()` و `no_grad()` یک عکس را پیش‌بینی می‌کند. `unsqueeze(0)` بُعد batch را اضافه می‌کند؛ بنابراین شکل ورودی از `[3, 224, 224]` به `[1, 3, 224, 224]` می‌رسد. سپس `softmax` امتیازهای خام را به سهم هر یک از هشت کلاس تبدیل می‌کند و `needs_review` را با آستانهٔ داخل checkpoint تعیین می‌کند.

برای دیدن حالتِ ارجاع به بازبینی انسانی، اینجا یک تصویر validation را که در قدم ۳۴ زیر آستانه قرار گرفت انتخاب می‌کنیم. **برچسب واقعی فقط برای کنترل ما در نوت‌بوک است**؛ در استفادهٔ واقعی، `predict.py` آن را نمی‌داند. مقایسهٔ پیش‌بینی با قدم ۲۸ بررسی می‌کند که اسکریپت همان مدل و ترتیب کلاس را به کار می‌گیرد. این سلول آموزشی است و هیچ وزنی را تغییر نمی‌دهد.

برای خوانایی، JSON را با عددهای اعشاریِ بدون `e-...` چاپ می‌کنیم و پایین آن همان مقادیر را به درصد نشان می‌دهیم. درصدها فقط قالب نمایشی‌اند؛ مقدارهای عددی JSON همچنان در بازهٔ صفر تا یک می‌مانند.

چون این نوت‌بوک قبلاً `predict.py` را import کرده بود، `importlib.reload(predict)` نسخهٔ به‌روزشدهٔ فایل را در همین کرنل دوباره می‌خواند؛ نیازی به اجرای مجدد آموزش نیست.


In [ ]:
import importlib
import predict

predict = importlib.reload(predict)  # Read the updated predict.py in this running kernel.

assert reviewed_indices, "There must be a validation image below the selected threshold."
demo_index = reviewed_indices[0]
demo_image_path = project_root / resnet_validation_rows[demo_index]["path"]
demo_result = predict.predict_image(demo_image_path)

expected_prediction = resnet_class_names[accuracy_model_pred[demo_index]]
assert demo_result["predicted_class"] == expected_prediction
assert abs(sum(demo_result["probabilities"].values()) - 1.0) < 1e-5
assert demo_result["needs_review"] is True

print("True validation label:", resnet_class_names[accuracy_model_true[demo_index]])
print("Prediction matches notebook:", demo_result["predicted_class"] == expected_prediction)
print("Prediction JSON (plain decimal numbers):")
print(predict.format_prediction_json(demo_result))
print("Probabilities as percentages for reading:")
for class_name, score in demo_result["probabilities"].items():
    print(f"{class_name}: {score * 100:.12f}%")


## ResNet18، قدم ۳۶: ثبت انتخاب نهایی در نتایج

در `reports/experiment_results.json`، اجرای fine-tuning و هر دو معیار validation ثبت شده‌اند؛ فیلد `selected_epoch` آن آزمایش هنوز checkpoint با **کمترین loss** را توصیف می‌کند. ما برای استفادهٔ نهایی، پس از مقایسهٔ یکسان، checkpoint با **بیشترین accuracy** را برگزیدیم. برای اینکه تاریخچهٔ آزمایش را پاک نکنیم و تصمیم نهایی هم مبهم نماند، یک بخش جدا به نام `final_selection` در همان فایل نتایج ثبت می‌کنیم.

این بخش، fingerprint تقسیم داده، epoch و معیار انتخاب، accuracy/loss/macro-F1، آستانهٔ ارجاع و خلاصهٔ خطاهای ارجاع‌شده را از محاسبات و checkpoint فعلی می‌گیرد. همچنین ثبت می‌کند که کامیون و کامیونت فعلاً جدا می‌مانند. این سلول فقط گزارش JSON را به‌روزرسانی می‌کند؛ هیچ عکس یا وزن خصوصی وارد گزارش نمی‌شود و مدلی آموزش نمی‌بیند.


In [ ]:
import json

results_path = project_root / "reports" / "experiment_results.json"
all_results = json.loads(results_path.read_text(encoding="utf-8"))
assert all_results["_split"]["current_fingerprint"] == resnet_manifest["fingerprint"]
assert reloaded_final_checkpoint["split_fingerprint"] == resnet_manifest["fingerprint"]

all_results["final_selection"] = {
    "experiment": "resnet18_layer4_finetune",
    "result_status": "current_split",
    "split_fingerprint": resnet_manifest["fingerprint"],
    "checkpoint": "checkpoints/resnet18_final.pt",
    "selection_metric": reloaded_final_checkpoint["selection_metric"],
    "selected_epoch": reloaded_final_checkpoint["selected_epoch"],
    "validation_images": len(accuracy_model_true),
    "validation_loss": reloaded_final_checkpoint["validation_loss"],
    "validation_accuracy": reloaded_final_checkpoint["validation_accuracy"],
    "validation_macro_f1": float(accuracy_model_macro_f1),
    "validation_mistakes": accuracy_model_mistakes,
    "review_threshold": reloaded_final_checkpoint["review_threshold"],
    "threshold_source": reloaded_final_checkpoint["threshold_source"],
    "threshold_validation": reloaded_final_checkpoint["threshold_validation"],
    "class_merge_decision": "Keep kamyun and kamyunet separate; review ambiguous labels.",
    "prediction_script": "predict.py",
}
results_path.write_text(
    json.dumps(all_results, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Final selection recorded in:", results_path)
print("Selected epoch:", all_results["final_selection"]["selected_epoch"])
print("Validation accuracy:", f'{all_results["final_selection"]["validation_accuracy"]:.1%}')
print("Review threshold:", all_results["final_selection"]["review_threshold"])


## CNN، قدم ۳۷: مبنای مشترک آزمایش‌های کنترل‌شده

بخش «Controlled ablations» صورت پروژه می‌خواهد **هر بار فقط یک عامل** را عوض کنیم. قدم ۳۷ یک غربال اولیهٔ ۱۰‌epochی بود. برای مقایسهٔ اصلیِ بعدی، با توجه به نتیجهٔ CNN سی‌epochیِ قبلی، بودجه را **۳۰ epoch برای هر روش** می‌گذاریم و آن اجرا را مبنا می‌گیریم. نسخهٔ مرجع ۳۰‌epochی قبلاً در همین نوت‌بوک اجرا و ذخیره شده است؛ augmentation، dropout، pooling، weight decay، scheduler، BCE و نمونه‌برداری متوازن را با همان بودجهٔ ۳۰ epoch مقایسه خواهیم کرد.

این اجرا با CNN سی‌epochیِ آموزشیِ قبلی فرق دارد. اگر عدد دقت اینجا متفاوت بود، قبل از نسبت‌دادن آن به روش جدید باید به بودجهٔ epoch توجه کنیم. تابع `run` در هر epoch train/validation loss و accuracy را محاسبه می‌کند، بهترین epoch را بر اساس validation accuracy انتخاب می‌کند، و معیارهای هر کلاس را در همان `reports/experiment_results.json` ثبت می‌کند. checkpoint داخل پوشهٔ خصوصی `checkpoints/` می‌ماند.

`importlib.reload` نسخهٔ جدید اسکریپت آزمایش‌ها را در کرنل فعلی می‌خواند؛ چون این ماژول قبلاً در نوت‌بوک import شده بود. این سلول **آموزش واقعیِ CNN مرجع** را انجام می‌دهد و ممکن است چند دقیقه طول بکشد.


In [ ]:
import importlib
import scripts.run_experiments as experiment_runner

experiment_runner = importlib.reload(experiment_runner)
experiment_runner.run("baseline", epochs=10)


## CNN، قدم ۳۸: اثر augmentation افقی

در این آزمایش تنها تفاوت با CNN مرجعِ ۳۰‌epochیِ قبلی، افزودن `RandomHorizontalFlip(p=0.5)` به **تصاویر آموزش** است: هر بار که تصویری خوانده می‌شود، با احتمال ۵۰٪ چپ و راست آن جابه‌جا می‌شود. برچسب خودرو عوض نمی‌شود. تصاویر validation هیچ flip تصادفی ندارند؛ بنابراین معیار ارزیابی پایدار می‌ماند.

بقیهٔ موارد—تقسیم داده، معماری CNN، seed، batch size، optimizer، learning rate و ۳۰ epoch—همان هستند. `run("augment", epochs=30)` وزن‌ها را از ابتدا آموزش می‌دهد و بهترین checkpoint را بر اساس validation accuracy در پوشهٔ خصوصی `checkpoints/` ذخیره می‌کند. نتیجه و معیارهای هر کلاس در همان فایل `reports/experiment_results.json` ثبت می‌شوند. پس از اجرا می‌توانیم بررسی کنیم این تنوع مصنوعی واقعاً به تعمیم مدل کمک کرده یا نه.

اجرای ۱۰‌epochیِ قدم ۳۷ فقط غربال اولیه بود؛ اینجا هر دو طرف مقایسه ۳۰ epoch فرصت دارند. ده epoch اول آن اجرا دقیقاً با ده epoch اول CNN قبلی یکسان شد، که بازتولید تنظیمات پایه را تأیید می‌کند.


In [ ]:
import json

experiment_runner.run("augment", epochs=30)

comparison_results = json.loads(
    (project_root / "reports" / "experiment_results.json").read_text()
)
plain_30 = comparison_results["cnn_baseline"]
augment_30 = comparison_results["augment"]
assert plain_30["split_fingerprint"] == augment_30["split_fingerprint"]
assert augment_30["epochs"] == 30
print("30-epoch CNN baseline: accuracy", f'{plain_30["accuracy"]:.1%}',
      "macro-F1", f'{plain_30["macro avg"]["f1-score"]:.3f}')
print("30-epoch augmentation: accuracy", f'{augment_30["accuracy"]:.1%}',
      "macro-F1", f'{augment_30["macro_f1"]:.3f}')


## CNN، قدم ۳۹: نتیجهٔ flip افقی

روی همان ۸۶۷ تصویر validation، CNN مرجعِ ۳۰‌epochی در بهترین epoch خود **۷۴۳ تصویر** را درست تشخیص داد: accuracy برابر `85.7%` و macro-F1 برابر `0.857`. نسخهٔ دارای flip افقی در بهترین epoch خود **۷۳۶ تصویر** را درست تشخیص داد: accuracy برابر `84.9%` و macro-F1 برابر `0.848`. پس این augmentation ساده در این اجرای کنترل‌شده بهبود کلی نداده است.

اثر برای همهٔ کلاس‌ها یکسان نیست: recall وانت از `0.850` به `0.881` افزایش یافت، اما precision کامیونت از `0.731` به `0.678` کاهش یافت. در پایان آموزش، train accuracy نزدیک `99.6%` بود، در حالی که validation accuracy `83.5%` و validation loss رو به افزایش بود؛ این نشانهٔ تعمیم ناکافی و خطاهای با اطمینان بالا در epochهای دیرتر است. نتیجه فقط دربارهٔ **flip افقی با این تنظیمات، این split و seed** است؛ از آن نمی‌توان نتیجه گرفت همهٔ روش‌های augmentation بد هستند. checkpoint منتخب این آزمایش مربوط به epoch ۲۳ است، نه epoch ۳۰.


## CNN، قدم ۴۰: آزمایش dropout با احتمال ۰٫۳

حالا فقط یک عامل دیگر را عوض می‌کنیم: در بخش classifier، بعد از `Flatten` و قبل از لایهٔ `Linear`، `nn.Dropout(p=0.3)` قرار می‌گیرد. هنگام **آموزش** در هر forward، بخشی از ویژگی‌ها به‌صورت تصادفی صفر می‌شوند تا مدل کمتر به تک‌ویژگی‌ها وابسته شود. هنگام **validation**، `model.eval()` این حذف تصادفی را خاموش می‌کند. dropout وزنِ قابل‌آموزش تازه‌ای اضافه نمی‌کند.

داده، معماری کانولوشن، optimizer، learning rate، seed و بودجهٔ ۳۰ epoch همان CNN مرجع هستند. `run("dropout", epochs=30)` یک CNN تازه می‌سازد و آموزش می‌دهد؛ سپس accuracy و macro-F1 آن را با CNN مرجع ۳۰‌epochی مقایسه می‌کنیم. این آزمایش هنوز دربارهٔ `p=0.5` نتیجه نمی‌دهد؛ اگر لازم باشد آن مقدار را هم جداگانه و با همین بودجه بررسی می‌کنیم.


In [ ]:
import json

experiment_runner.run("dropout", epochs=30)

comparison_results = json.loads(
    (project_root / "reports" / "experiment_results.json").read_text()
)
plain_30 = comparison_results["cnn_baseline"]
dropout_30 = comparison_results["dropout"]
assert plain_30["split_fingerprint"] == dropout_30["split_fingerprint"]
assert dropout_30["epochs"] == 30
print("30-epoch CNN baseline: accuracy", f'{plain_30["accuracy"]:.1%}',
      "macro-F1", f'{plain_30["macro avg"]["f1-score"]:.3f}')
print("30-epoch dropout p=0.3: accuracy", f'{dropout_30["accuracy"]:.1%}',
      "macro-F1", f'{dropout_30["macro_f1"]:.3f}')


## CNN، قدم ۴۱: نتیجهٔ dropout با احتمال ۰٫۳

در بهترین epoch از ۳۰ epoch، مدل `Dropout(p=0.3)` تعداد **۷۳۴ از ۸۶۷** تصویر validation را درست تشخیص داد: accuracy برابر `84.7%` و macro-F1 برابر `0.845`. CNN مرجع **۷۴۳ تصویر** را درست تشخیص داد: `85.7%` و `0.857`. بنابراین این مقدار dropout در این اجرا ۹ پاسخ درست کمتر داد؛ بهترین epoch آن ۱۱ بود، در برابر epoch ۲۴ مرجع. هر دو مدل فرصت ۳۰‌epochی و معیار انتخاب یکسان داشتند.

تغییرات کلاسی یکنواخت نیستند: F1 وانت از `0.866` به `0.880` بهتر شد، ولی F1 مینی‌بوس از `0.891` به `0.836` و کامیون از `0.779` به `0.747` افت کرد. در epochهای دیرتر train accuracy نزدیک `99%` ماند و validation loss افزایش یافت؛ پس `p=0.3` به‌تنهایی مشکل تعمیم را حل نکرده است. این نتیجه برای همین split، seed و محل قرارگرفتن Dropout است، نه حکم کلی دربارهٔ Dropout.


## CNN، قدم ۴۲: dropout با احتمال ۰٫۵

برای کامل‌کردن آزمایش تک‌عاملی dropout، مقدار `p=0.5` را هم امتحان می‌کنیم. تفاوت با مرجع فقط این است که هنگام آموزش هر بار **نیمی از ویژگی‌های خروجی Flatten به‌صورت تصادفی صفر می‌شوند**؛ هنگام validation، `eval()` dropout را غیرفعال می‌کند. `p=0.3` و `p=0.5` دو اجرای مستقل‌اند و هیچ‌کدام آموزش دیگری را ادامه نمی‌دهند.

پس از تغییر `scripts/run_experiments.py`، با `importlib.reload` نسخهٔ جدید را در کرنل می‌خوانیم. مدل تازه ۳۰ epoch آموزش می‌بیند و در پایان، دقت و macro-F1 هر سه حالت `p=0`، `p=0.3` و `p=0.5` روی **همان validation** کنار هم چاپ می‌شوند.


In [ ]:
import importlib
import json

experiment_runner = importlib.reload(experiment_runner)
experiment_runner.run("dropout_05", epochs=30)

comparison_results = json.loads(
    (project_root / "reports" / "experiment_results.json").read_text()
)
plain_30 = comparison_results["cnn_baseline"]
dropout_03 = comparison_results["dropout"]
dropout_05 = comparison_results["dropout_05"]
assert len({
    plain_30["split_fingerprint"],
    dropout_03["split_fingerprint"],
    dropout_05["split_fingerprint"],
}) == 1
assert dropout_03["epochs"] == dropout_05["epochs"] == 30
print("p=0.0: accuracy", f'{plain_30["accuracy"]:.1%}',
      "macro-F1", f'{plain_30["macro avg"]["f1-score"]:.3f}')
print("p=0.3: accuracy", f'{dropout_03["accuracy"]:.1%}',
      "macro-F1", f'{dropout_03["macro_f1"]:.3f}')
print("p=0.5: accuracy", f'{dropout_05["accuracy"]:.1%}',
      "macro-F1", f'{dropout_05["macro_f1"]:.3f}')


## CNN، قدم ۴۳: نتیجهٔ سه مقدار dropout

در این تقسیم و با ۳۰ epoch برای هر اجرا، بهترین checkpointها چنین بودند: `p=0` با **۷۴۳/۸۶۷** پاسخ درست (`85.7%`، macro-F1=`0.857`)، `p=0.3` با **۷۳۴/۸۶۷** (`84.7%`، `0.845`)، و `p=0.5` با **۷۳۷/۸۶۷** (`85.0%`، `0.850`). بهترین epochها به‌ترتیب ۲۴، ۱۱ و ۱۵ بودند. بر اساس دقت و macro-F1 کلی، **نسخهٔ بدون dropout در این آزمایش بهتر بود**.

اثر dropout یکسان یا یکنواخت نیست: با `p=0.5`، F1 تاکسی از `0.954` به `0.967` و مینی‌بوس از `0.891` به `0.900` رسید، ولی F1 کامیون از `0.779` به `0.726` و کامیونت از `0.752` به `0.725` افت کرد. حتی با dropout، train accuracy در پایان نزدیک `99%` شد و validation loss در epochهای دیرتر افزایش یافت. این نتیجه فقط دربارهٔ این محل قرارگیری dropout، یک split و یک seed است؛ از آن نمی‌توان قانون کلی «dropout بد است» ساخت.


## CNN، قدم ۴۴: MaxPool در برابر AvgPool

قبلاً در CNN از `MaxPool2d(2)` استفاده کردیم: در هر پنجرهٔ `2×2`، **بزرگ‌ترین** عدد نقشهٔ ویژگی باقی می‌ماند. در این آزمایش همان پنجره را با `AvgPool2d(2)` عوض می‌کنیم که **میانگین چهار عدد** را نگه می‌دارد. اندازهٔ خروجی هر دو یکی است، اما اطلاعاتی که حفظ می‌کنند فرق دارد؛ هیچ‌کدام وزنِ قابل‌آموزش تازه‌ای ندارند.

`run("avg_pool", epochs=30)` یک CNN تازه با AvgPool می‌سازد. داده، کانولوشن‌ها، dropout=`0`، optimizer، seed و بودجهٔ آموزش با CNN مرجع برابرند؛ بنابراین تفاوت اصلی همین روش خلاصه‌کردن نقشهٔ ویژگی است. بعد از اجرا accuracy و macro-F1 بهترین checkpoint دو روش را روی همان validation می‌خوانیم.


In [ ]:
import json

experiment_runner.run("avg_pool", epochs=30)

comparison_results = json.loads(
    (project_root / "reports" / "experiment_results.json").read_text()
)
max_pool_30 = comparison_results["cnn_baseline"]
avg_pool_30 = comparison_results["avg_pool"]
assert max_pool_30["split_fingerprint"] == avg_pool_30["split_fingerprint"]
assert avg_pool_30["epochs"] == 30
print("MaxPool baseline: accuracy", f'{max_pool_30["accuracy"]:.1%}',
      "macro-F1", f'{max_pool_30["macro avg"]["f1-score"]:.3f}')
print("AvgPool experiment: accuracy", f'{avg_pool_30["accuracy"]:.1%}',
      "macro-F1", f'{avg_pool_30["macro_f1"]:.3f}')


## CNN، قدم ۴۵: نتیجهٔ MaxPool و AvgPool

با همان ۳۰ epoch و validation یکسان، CNN مرجع با `MaxPool2d(2)` تعداد **۷۴۳ از ۸۶۷** تصویر را درست تشخیص داد (`85.7%`، macro-F1=`0.857`). نسخهٔ `AvgPool2d(2)` تعداد **۷۲۸ تصویر** را درست تشخیص داد (`84.0%`، `0.839`). بهترین epoch AvgPool برابر ۱۱ بود، در حالی که مرجع در epoch ۲۴ به بهترین accuracy رسید. پس برای این CNN و این داده، **MaxPool انتخاب بهتری برای نتیجهٔ کلی** است.

بزرگ‌ترین افت کلاسی در کامیون دیده شد: recall از `0.761` به `0.697` و F1 از `0.779` به `0.710` رسید. در مقابل F1 آمبولانس از `0.873` به `0.889` و وانت از `0.866` به `0.876` افزایش یافت. از epoch ۱۴ به بعد train accuracy تقریباً `100%` بود، اما validation accuracy حدود `82–83%` ماند و validation loss تا `1.2607` در epoch ۳۰ بالا رفت؛ AvgPool در این اجرا مشکل تعمیم را حل نکرد. این نتیجه یک مقایسهٔ تجربی است، نه اینکه MaxPool برای همهٔ مسائل بهتر باشد.


## CNN، قدم ۴۶: اثر weight decay

حالا به‌جای تغییر داده یا pooling، فقط **شدت محدودسازی وزن‌ها** را تغییر می‌دهیم. آزمایش مرجع `weight_decay=0` داشت؛ این اجرا `weight_decay=1e-4`، یعنی `0.0001`، را به همان optimizer `Adam` می‌دهد. این تنظیم در به‌روزرسانی وزن‌ها، بزرگ‌شدن بی‌رویهٔ آن‌ها را پرهزینه‌تر می‌کند و شاید به تعمیم کمک کند. مدل و split، seed، batch size، learning rate و بودجهٔ ۳۰ epoch ثابت می‌مانند.

در `run_experiments.py` نام آزمایش `"weight_decay"` است. بعد از آموزش یک CNN تازه، accuracy و macro-F1 بهترین checkpoint را با نسخهٔ مرجع مقایسه می‌کنیم. فقط بهتر یا بدتر شدن روی validation فعلی را گزارش می‌کنیم؛ علت قطعی یا نتیجهٔ تضمینی برای دادهٔ پنهان از آن درنمی‌آید.


In [ ]:
import json

experiment_runner.run("weight_decay", epochs=30)

comparison_results = json.loads(
    (project_root / "reports" / "experiment_results.json").read_text()
)
plain_30 = comparison_results["cnn_baseline"]
decay_30 = comparison_results["weight_decay"]
assert plain_30["split_fingerprint"] == decay_30["split_fingerprint"]
assert decay_30["epochs"] == 30
assert decay_30["configuration"]["weight_decay"] == 1e-4
print("Adam, weight decay=0: accuracy", f'{plain_30["accuracy"]:.1%}',
      "macro-F1", f'{plain_30["macro avg"]["f1-score"]:.3f}')
print("Adam, weight decay=1e-4: accuracy", f'{decay_30["accuracy"]:.1%}',
      "macro-F1", f'{decay_30["macro_f1"]:.3f}')


## CNN، قدم ۴۷: تحلیل weight decay

در مقایسهٔ ۳۰-epochی روی همان ۸۶۷ تصویر validation، مدل پایه با `weight_decay=0` به accuracy برابر **۸۵٫۷٪** (۷۴۳ پاسخ درست) و macro-F1 برابر **۰٫۸۵۷** رسید. مدل با `weight_decay=1e-4` به accuracy برابر **۸۵٫۱٪** (۷۳۸ پاسخ درست) و macro-F1 برابر **۰٫۸۴۹** رسید؛ یعنی در این اجرا **۵ پاسخ درست کمتر**. معماری، تقسیم داده، seed، batch size، optimizer از نوع Adam و تعداد epoch ثابت بودند.

در آزمایش weight decay، بهترین accuracy در epoch ۲۹ ثبت شد؛ آن زمان validation loss حدود `0.976` بود. کمترین validation loss این اجرا در epoch ۴ و حدود `0.631` بود، ولی accuracy آن epoch فقط `81.0%` بود. پس «بهترین دقت» و «کمترین loss» الزاماً در یک epoch رخ نمی‌دهند. در epochهای پایانی train accuracy به `100%` رسید، در حالی که validation accuracy حدود `85%` ماند و validation loss نسبت به ابتدای آموزش بالا رفت؛ این نشانهٔ شکاف تعمیم است.

نتیجهٔ محدود این آزمایش: `weight_decay=1e-4` به‌تنهایی و با این تنظیم‌ها دقت validation را بهتر نکرد. این نتیجه به معنی بی‌فایده‌بودن weight decay در همهٔ مقدارها یا در ترکیب با روش‌های دیگر نیست.


## CNN، قدم ۴۸: آزمایش scheduler

تا اینجا learning rate مربوط به Adam در تمام epochها `0.001` ثابت بود. حالا فقط یک **scheduler** اضافه می‌کنیم: `ReduceLROnPlateau` پس از هر epoch مقدار validation loss را می‌بیند. اگر loss چند epoch بهتر نشود، learning rate را نصف می‌کند (`factor=0.5` و `patience=2`). برای نمونه، مقدار `0.001` می‌تواند به `0.0005` برسد. هدف این است که مدل پس از توقف پیشرفت، با قدم‌های کوچک‌تر وزن‌ها را تنظیم کند. scheduler خودش وزن‌ها را مستقیماً تغییر نمی‌دهد؛ مقدار learning rate در optimizer را تغییر می‌دهد.

در این آزمایش، معماری CNN، داده، seed، batch size، Adam، `weight_decay=0`، `dropout=0` و ۳۰ epoch مانند مرجع می‌مانند. فقط برنامهٔ learning rate فرق دارد. checkpoint همچنان بر اساس **بیشترین validation accuracy** انتخاب می‌شود؛ validation loss فقط سیگنال تصمیم‌گیری scheduler است. اجرای زیر یک مدل تازه را از ابتدا آموزش می‌دهد و نتیجه‌اش را با مرجع مقایسه می‌کند.


In [ ]:
import importlib
import json

experiment_runner = importlib.reload(experiment_runner)
experiment_runner.run("scheduler", epochs=30)

comparison_results = json.loads(
    (project_root / "reports" / "experiment_results.json").read_text()
)
plain_30 = comparison_results["cnn_baseline"]
scheduler_30 = comparison_results["scheduler"]
assert plain_30["split_fingerprint"] == scheduler_30["split_fingerprint"]
assert scheduler_30["epochs"] == 30

print("Fixed learning rate: accuracy", f'{plain_30["accuracy"]:.1%}',
      "macro-F1", f'{plain_30["macro avg"]["f1-score"]:.3f}')
print("Scheduler: accuracy", f'{scheduler_30["accuracy"]:.1%}',
      "macro-F1", f'{scheduler_30["macro_f1"]:.3f}')
print("Scheduler best epoch:", scheduler_30["best_epoch"])

# history records the learning rate used during each epoch, before the scheduler's end-of-epoch update.
previous_lr = None
for row in scheduler_30["history"]:
    lr = row["learning_rates"][0]
    if lr != previous_lr:
        print(f'Epoch {row["epoch"]:02d} starts with learning rate {lr:.6f}')
        previous_lr = lr


## CNN، قدم ۴۹: نتیجهٔ scheduler و معنی loss

در هر دو اجرای ۳۰-epochی روی همان split، بهترین validation accuracy برابر **۸۵٫۷٪** (۷۴۳ پاسخ درست از ۸۶۷ تصویر) و macro-F1 حدود **۰٫۸۵۷** بود؛ هر دو checkpoint نخستین‌بار در epoch ۲۴ به این دقت رسیدند. بنابراین scheduler در این اجرا **تعداد پاسخ‌های درست را بیشتر نکرد**.

| اجرا | validation loss در epoch انتخاب‌شده بر اساس دقت | کمترین validation loss کل اجرا |
|---|---:|---:|
| learning rate ثابت | `0.9704` در epoch ۲۴ | `0.6247` در epoch ۴ |
| scheduler | `0.7738` در epoch ۲۴ | `0.6247` در epoch ۴ |

پس loss مربوط به **checkpoint انتخاب‌شده** در اجرای scheduler پایین‌تر بود، با اینکه accuracy یکسان است. accuracy فقط درست یا نادرست‌بودن کلاس اول را می‌شمارد؛ CrossEntropyLoss به احتمال/امتیاز داده‌شده به کلاس واقعی هم حساس است. این تفاوت loss نشان می‌دهد کیفیت امتیازهای مدل‌ها یکسان نیست، ولی بدون بررسی جداگانهٔ اطمینان‌ها نمی‌توان علت جزئی آن را قطعی دانست.

scheduler کمترین loss را در epoch ۴ دید. پس از چند epoch بدون بهبود loss، در پایان epoch ۷ learning rate را از `0.001` به `0.0005` کاهش داد؛ مقدار تازه از epoch ۸ برای آموزش استفاده شد. در پایان epoch ۱۰ دوباره نصف شد و این روند ادامه یافت. هرچند accuracy بعداً بهتر شد، scheduler هنوز **loss** را پایش می‌کرد، نه accuracy را. در epochهای پایانی train accuracy به `100%` رسید و validation loss دوباره افزایش یافت؛ کاهش learning rate این شکاف تعمیم را کاملاً از بین نبرد.


## CNN، قدم ۵۰: چرا batchهای متعادل را آزمایش می‌کنیم؟

در دادهٔ آموزش نسبتاً متعادل، `shuffle=True` و batch متعادل ممکن است تفاوت زیادی نشان ندهند. صورت پروژه می‌خواهد این اثر را در یک **نامتوازنی شبیه‌سازی‌شده و تکرارپذیر** بسنجیم. برای این آزمایش خاص، فقط تصاویر منبع `labelled` از بخش train را برمی‌داریم؛ تصاویر `unclean` و نیسان را در ساخت نامتوازنی دخالت نمی‌دهیم. این تغییر **فقط به این دو اجرای آزمایشی مربوط است** و تقسیم اصلی داده و validation را عوض نمی‌کند.

با seed ثابت، حدود یک‌چهارم تصاویر برچسب‌خوردهٔ `kamyun` و `kamyunet` را نگه می‌داریم و تصاویر شش کلاس دیگر را نگه می‌داریم. موقعیت نمونه‌های نگه‌داشته‌شده در فایل خصوصی و ignore‌شدهٔ `reports/imbalance_indices.json` ثبت می‌شود. هر دو اجرا دقیقاً همین تصاویر را می‌بینند؛ تفاوت فقط روش ساخت batch است.

در روش معمولی، `shuffle=True` فقط ترتیب کل تصاویر را به‌هم می‌زند و تضمین نمی‌کند هر batch از هر کلاس چند تصویر داشته باشد. در `BalancedBatchSampler`، batch سی‌ودوتایی شامل **۴ تصویر از هر یک از ۸ کلاس** است. ابتدا نمونه‌های هر کلاس را مخلوط می‌کنیم و بدون تکرار برمی‌داریم؛ اگر نمونه‌های یک کلاس تمام شوند، همان کلاس را دوباره مخلوط و استفاده می‌کنیم. بنابراین کلاس‌های کم‌نمونه بیشتر دیده می‌شوند. ابتدا سلول بررسی داده و یک batch را اجرا کن، سپس دو آموزش ۳۰-epochی را.


In [ ]:
import importlib
from collections import Counter

experiment_runner = importlib.reload(experiment_runner)
manifest, train_rows, val_rows = experiment_runner.manifest_data()
shuffle_loader, _, shuffle_rows = experiment_runner.make_loaders(
    manifest, train_rows, val_rows, "imbalance_shuffle"
)
balanced_loader, _, balanced_rows = experiment_runner.make_loaders(
    manifest, train_rows, val_rows, "balanced_batches"
)

assert [row["path"] for row in shuffle_rows] == [row["path"] for row in balanced_rows]
assert all(row["source"] == "labelled" for row in shuffle_rows)
print("Training rows shared by both methods:", len(shuffle_rows))
print("Validation rows unchanged:", len(val_rows))
print("Training class counts:", dict(sorted(Counter(row["label"] for row in shuffle_rows).items())))

first_batch_indices = next(iter(balanced_loader.batch_sampler))
first_batch_counts = Counter(balanced_rows[index]["label"] for index in first_batch_indices)
print("First balanced batch:", dict(sorted(first_batch_counts.items())))
assert len(first_batch_indices) == 32
assert set(first_batch_counts.values()) == {4}


In [ ]:
import importlib
import json

# Load the latest runner even if the preview cell was skipped.
experiment_runner = importlib.reload(experiment_runner)
manifest, train_rows, val_rows = experiment_runner.manifest_data()
_, _, shuffled_rows = experiment_runner.make_loaders(
    manifest, train_rows, val_rows, "imbalance_shuffle"
)
_, _, balanced_rows = experiment_runner.make_loaders(
    manifest, train_rows, val_rows, "balanced_batches"
)
assert len(shuffled_rows) == 1512
assert [row["path"] for row in shuffled_rows] == [row["path"] for row in balanced_rows]
assert all(row["source"] == "labelled" for row in shuffled_rows)
print("Verified training pool:", len(shuffled_rows), "labelled images; validation:", len(val_rows))

experiment_runner.run("imbalance_shuffle", epochs=30)
experiment_runner.run("balanced_batches", epochs=30)

comparison_results = json.loads(
    (project_root / "reports" / "experiment_results.json").read_text()
)
shuffled = comparison_results["imbalance_shuffle"]
balanced = comparison_results["balanced_batches"]
assert shuffled["split_fingerprint"] == balanced["split_fingerprint"]
assert shuffled["training_images"] == balanced["training_images"] == 1512
assert shuffled["validation_images"] == balanced["validation_images"]
assert shuffled["configuration"]["imbalance_source"] == "labelled_only"
assert balanced["configuration"]["imbalance_source"] == "labelled_only"
for result in (shuffled, balanced):
    print(result["experiment"], "accuracy", f'{result["accuracy"]:.1%}',
          "macro-F1", f'{result["macro_f1"]:.3f}',
          "best epoch", result["best_epoch"])
    for label in ("kamyun", "kamyunet"):
        print(" ", label, "recall", f'{result["per_class"][label]["recall"]:.1%}')


## CNN، قدم ۵۲: تحلیل batch متعادل در دادهٔ نامتوازن

هر دو مدل روی **همان ۱۵۱۲ تصویر برچسب‌خورده** (بدون `unclean` و نیسان در training این آزمایش) و همان **۸۶۷ تصویر validation**، با معماری یکسان و بودجهٔ ۳۰ epoch آموزش دیدند. تفاوت فقط روش نمونه‌برداری batch بود.

| روش | بهترین epoch بر اساس accuracy | پاسخ درست / ۸۶۷ | accuracy | macro-F1 |
|---|---:|---:|---:|---:|
| `shuffle=True` | ۲۴ | ۶۵۴ | ۷۵٫۴٪ | ۰٫۷۳۴ |
| `BalancedBatchSampler` | ۱۱ | ۶۶۳ | ۷۶٫۵٪ | ۰٫۷۴۹ |

پس batch متعادل در این شبیه‌سازی **۹ پیش‌بینی درست بیشتر** و macro-F1 حدود `0.015` بالاتر داد. recall کامیون از **۴۹٫۵٪ (۵۴/۱۰۹)** به **۵۱٫۴٪ (۵۶/۱۰۹)** و کامیونت از **۳۲٫۴٪ (۳۳/۱۰۲)** به **۳۶٫۳٪ (۳۷/۱۰۲)** رسید. این پیشرفت محدود است؛ برای مثال recall وانت از **۸۶٫۲٪** به **۸۱٫۹٪** و تاکسی از **۹۶٫۳٪** به **۹۴٫۴٪** کاهش یافت.

هر دو اجرا در بهترین epoch خود train accuracy برابر `100%` داشتند، ولی validation accuracy بسیار پایین‌تر بود؛ پس مدل همچنان تعمیم محدودی دارد. دقت این دو اجرای نامتوازن را نباید مستقیم به‌عنوان اثر sampler با CNN پایهٔ **۳۴۶۹تصویری** مقایسه کرد: آن مقایسه هم‌زمان اندازه و ترکیب دادهٔ آموزش را تغییر می‌دهد. نتیجهٔ مربوط به sampler فقط مقایسهٔ همین دو اجرای **۱۵۱۲تصویری** است.


## CNN، قدم ۵۳: CrossEntropyLoss در برابر BCEWithLogitsLoss

مسئلهٔ ما **تک‌برچسبی و هشت‌کلاسه** است: هر تصویر دقیقاً یک برچسب درست دارد. `CrossEntropyLoss` این کلاس‌ها را رقیب هم می‌بیند؛ خروجی `softmax` مجموعاً ۱ می‌شود. حالا برای هدف آموزشی، همان CNN را با `BCEWithLogitsLoss` آموزش می‌دهیم. BCE هر خروجی را یک تصمیم بله/خیر **مستقل** فرض می‌کند. برچسب عددی مانند `2` را با `one_hot` به بردار هشت‌تایی مانند `[0, 0, 1, 0, 0, 0, 0, 0]` تبدیل می‌کنیم و با `.float()` به نوع موردنیاز BCE می‌رسانیم. خود `BCEWithLogitsLoss` بخش sigmoid را در محاسبهٔ loss دارد؛ پیش از آن sigmoid نمی‌زنیم.

برای گزارش accuracy و ماتریس خطا، در هر دو روش کلاسِ دارای بزرگ‌ترین logit را با `argmax` برمی‌داریم. با وجود این، sigmoidهای هشت خروجی BCE مستقل‌اند و لازم نیست مجموعشان ۱ شود؛ پس آن‌ها را مانند توزیع احتمال هشت‌کلاسهٔ softmax تفسیر نمی‌کنیم.

در اجرای بعدی، split، ۳۴۶۹ تصویر training، ۸۶۷ تصویر validation، معماری، seed، batch size، Adam، learning rate و ۳۰ epoch همان مرجع می‌مانند؛ **فقط loss تغییر می‌کند**. روند loss هر روش را می‌شود بررسی کرد، ولی عدد خام BCE و CrossEntropy را مستقیم با هم رتبه‌بندی نمی‌کنیم، چون تعریف و مقیاسشان متفاوت است. معیارهای مشترکِ مقایسه accuracy، macro-F1 و recall هر کلاس‌اند.


In [ ]:
import importlib
import json
import torch

experiment_runner = importlib.reload(experiment_runner)
experiment_runner.run("bce", epochs=30)

comparison_results = json.loads(
    (project_root / "reports" / "experiment_results.json").read_text()
)
ce_30 = comparison_results["cnn_baseline"]
bce_30 = comparison_results["bce"]
ce_checkpoint = torch.load(
    project_root / "checkpoints" / "cnn_baseline_best.pt",
    map_location="cpu",
    weights_only=True,
)
assert len(ce_checkpoint["history"]) == bce_30["epochs"] == 30
assert ce_30["split_fingerprint"] == bce_30["split_fingerprint"]
assert ce_checkpoint["split_fingerprint"] == bce_30["split_fingerprint"]
assert bce_30["training_images"] == 3469
assert bce_30["validation_images"] == 867

print("CrossEntropy: accuracy", f'{ce_30["accuracy"]:.1%}',
      "macro-F1", f'{ce_30["macro avg"]["f1-score"]:.3f}',
      "best epoch", ce_30["best_epoch"])
print("BCEWithLogits: accuracy", f'{bce_30["accuracy"]:.1%}',
      "macro-F1", f'{bce_30["macro_f1"]:.3f}',
      "best epoch", bce_30["best_epoch"])
print("Recall by class (CrossEntropy -> BCE):")
for label in experiment_runner.CLASS_LABELS:
    ce_recall = ce_30[label]["recall"]
    bce_recall = bce_30["per_class"][label]["recall"]
    print(f"  {label}: {ce_recall:.1%} -> {bce_recall:.1%}")


## CNN، قدم ۵۴: تحلیل اولیهٔ CrossEntropy و BCE

هر دو مدل روی **۳۴۶۹ تصویر آموزش** و **۸۶۷ تصویر validation** با همان معماری، seed، optimizer و ۳۰ epoch اجرا شدند؛ فقط تابع خطا فرق داشت. checkpointها بر اساس بیشترین validation accuracy انتخاب شده‌اند.

| تابع خطا | بهترین epoch | درست از ۸۶۷ | accuracy | macro-F1 |
|---|---:|---:|---:|---:|
| CrossEntropyLoss | ۲۴ | ۷۴۳ | ۸۵٫۷٪ | ۰٫۸۵۷ |
| BCEWithLogitsLoss | ۱۴ | ۷۴۱ | ۸۵٫۵٪ | ۰٫۸۵۴ |

BCE در این اجرا **۲ تصویر کمتر** را درست تشخیص داد. اثر آن بر کلاس‌ها یکسان نبود: recall کامیون از **۷۶٫۱٪** به **۷۱٫۶٪** و کامیونت از **۷۷٫۵٪** به **۷۴٫۵٪** افت کرد، در حالی که recall آمبولانس از **۸۷٫۸٪** به **۹۰٫۲٪** و تاکسی از **۹۵٫۴٪** به **۹۷٫۲٪** رسید. بنابراین فقط اختلاف دقت کلی برای انتخاب کافی نیست؛ الگوی خطا نیز مهم است.

کمترین validation loss اجرای BCE حدود `0.133` در epoch ۵ بود، با accuracy برابر `84.0%`. بهترین accuracy آن در epoch ۱۴ رخ داد، زمانی که loss حدود `0.209` بود. از epoch ۱۲ به بعد train accuracy تقریباً `100%` ماند، اما validation loss تا حدود `0.285` در epoch ۳۰ بالا رفت؛ این شکاف با بیش‌برازش سازگار است. عددهای BCE را با loss خام CrossEntropy مقایسه نمی‌کنیم، چون BCE میانگین هشت مسئلهٔ بله/خیر را حساب می‌کند و CrossEntropy یک مسئلهٔ هشت‌کلاسه را.

در قدم بعدی باید رفتار scoreها و اطمینان مدل‌ها را روی تصاویر مشترک بررسی کنیم: خروجی sigmoid برای BCE هشت مقدار مستقل می‌دهد و مجموعشان الزاماً ۱ نیست؛ softmax برای CrossEntropy یک توزیع هشت‌کلاسه با مجموع ۱ می‌دهد.


## CNN، قدم ۵۵: بررسی scoreها و اختلاف خطاهای CE و BCE

اکنون **دو checkpoint ذخیره‌شده** را روی همان ۸۶۷ تصویر validation اجرا می‌کنیم؛ آموزش تازه‌ای انجام نمی‌شود. برای هر تصویر، هر مدل هشت logit می‌دهد. روی logitهای مدل CrossEntropy، `softmax(dim=1)` توزیعی با مجموع ۱ می‌سازد. روی logitهای BCE، `sigmoid()` هشت مقدار مستقل می‌سازد؛ ممکن است هیچ‌کدام یا چندتایشان از `0.5` بزرگ‌تر باشند و مجموعشان الزاماً ۱ نیست. با این وجود برای گزارش هشت‌کلاسه، بزرگ‌ترین logit را با `argmax(dim=1)` انتخاب می‌کنیم.

سلول زیر علاوه بر مجموع خروجی‌ها، تعداد تصاویر درست/نادرستِ مشترک و اختلاف نظر دو مدل را چاپ می‌کند. «بزرگ‌ترین sigmoid» BCE را فقط برای بررسی رفتار score گزارش می‌کنیم؛ آن را احتمالِ نرمال‌شده یا اطمینانِ کالیبره‌شدهٔ یک پیش‌بینی هشت‌کلاسه نمی‌نامیم. `eval()` رفتار inference مدل را تنظیم می‌کند و `torch.no_grad()` ثبت محاسبات گرادیان را در این ارزیابی غیرفعال می‌کند.
نکتهٔ بارگذاری: مدل آموزشیِ اولیه `Flatten → Linear` داشت، ولی runner یک `Dropout(0)` بدون پارامتر بین آن‌ها دارد. بنابراین هنگام خواندن checkpoint اولیه فقط نام دو پارامتر `classifier.1` به `classifier.2` منتقل می‌شود؛ مقدار وزن‌ها تغییر نمی‌کند.


In [ ]:
import importlib
import torch

experiment_runner = importlib.reload(experiment_runner)
manifest, train_rows, val_rows = experiment_runner.manifest_data()
_, shared_val_loader, _ = experiment_runner.make_loaders(
    manifest, train_rows, val_rows, "baseline"
)
device = experiment_runner.device_for_run()

ce_checkpoint = torch.load(
    project_root / "checkpoints" / "cnn_baseline_best.pt",
    map_location="cpu", weights_only=True,
)
bce_checkpoint = torch.load(
    project_root / "checkpoints" / "bce_best.pt",
    map_location="cpu", weights_only=True,
)
assert ce_checkpoint["split_fingerprint"] == bce_checkpoint["split_fingerprint"] == manifest["fingerprint"]
assert ce_checkpoint["class_to_idx"] == bce_checkpoint["class_to_idx"] == manifest["class_to_idx"]

ce_model, _ = experiment_runner.make_model("baseline", device)
bce_model, _ = experiment_runner.make_model("bce", device)

# The original notebook used Flatten -> Linear; the runner uses
# Flatten -> Dropout(0) -> Linear. Dropout(0) has no parameters.
ce_state = dict(ce_checkpoint["model_state_dict"])
assert "classifier.1.weight" in ce_state and "classifier.2.weight" not in ce_state
ce_state["classifier.2.weight"] = ce_state.pop("classifier.1.weight")
ce_state["classifier.2.bias"] = ce_state.pop("classifier.1.bias")
ce_model.load_state_dict(ce_state)
bce_model.load_state_dict(bce_checkpoint["model_state_dict"])
ce_model.eval()
bce_model.eval()

true_batches, ce_batches, bce_batches = [], [], []
with torch.no_grad():
    for images, labels in shared_val_loader:
        images = images.to(device)
        ce_batches.append(ce_model(images).cpu())
        bce_batches.append(bce_model(images).cpu())
        true_batches.append(labels)

true_labels = torch.cat(true_batches)
ce_logits = torch.cat(ce_batches)
bce_logits = torch.cat(bce_batches)
assert ce_logits.shape == bce_logits.shape == (len(val_rows), 8)

ce_probabilities = ce_logits.softmax(dim=1)
bce_independent_scores = bce_logits.sigmoid()
ce_predictions = ce_logits.argmax(dim=1)
bce_predictions = bce_logits.argmax(dim=1)
ce_correct = ce_predictions == true_labels
bce_correct = bce_predictions == true_labels
assert int(ce_correct.sum()) == round(ce_checkpoint["best_val_accuracy"] * len(val_rows))
assert int(bce_correct.sum()) == round(bce_checkpoint["validation_accuracy"] * len(val_rows))
print("Saved checkpoint accuracies reproduced on validation.")

print("Softmax sum range:",
      f'{ce_probabilities.sum(dim=1).min():.3f}',
      "to", f'{ce_probabilities.sum(dim=1).max():.3f}')
print("BCE sigmoid sum range:",
      f'{bce_independent_scores.sum(dim=1).min():.3f}',
      "to", f'{bce_independent_scores.sum(dim=1).max():.3f}')
positive_counts = (bce_independent_scores > 0.5).sum(dim=1)
print("BCE images with zero / one / multiple scores above 0.5:",
      int((positive_counts == 0).sum()),
      int((positive_counts == 1).sum()),
      int((positive_counts > 1).sum()))
print("Both correct:", int((ce_correct & bce_correct).sum()))
print("Only CE correct:", int((ce_correct & ~bce_correct).sum()))
print("Only BCE correct:", int((~ce_correct & bce_correct).sum()))
print("Both wrong:", int((~ce_correct & ~bce_correct).sum()))
print("Different predicted classes:", int((ce_predictions != bce_predictions).sum()))

for name, scores, correct in (
    ("CE max softmax", ce_probabilities.max(dim=1).values, ce_correct),
    ("BCE max sigmoid", bce_independent_scores.max(dim=1).values, bce_correct),
):
    print(name, "mean on correct / wrong images:",
          f'{scores[correct].mean():.3f}', f'{scores[~correct].mean():.3f}')

example_index = next((i for i, count in enumerate(positive_counts.tolist()) if count > 1), 0)
print("Example index:", example_index, "true class:",
      experiment_runner.CLASS_LABELS[true_labels[example_index]])
print("CE softmax:", [round(float(x), 3) for x in ce_probabilities[example_index]])
print("BCE sigmoid:", [round(float(x), 3) for x in bce_independent_scores[example_index]])


## CNN، قدم ۵۶: معنی scoreها و هم‌پوشانی خطاها

بارگذاری هر دو checkpoint تأیید شد و تعداد پاسخ‌های درستِ ذخیره‌شده روی همان ۸۶۷ تصویر validation بازتولید شد. مجموع هشت مقدار `softmax` برای CE در همهٔ تصاویر تقریباً `1.000` بود. مجموع هشت مقدار `sigmoid` برای BCE از عددی بسیار کوچک (در چاپ سه‌رقمی `0.000`) تا حدود `1.995` تغییر کرد؛ پس خروجی sigmoidها یک توزیع هشت‌کلاسه با مجموع ۱ نیست. مقدارهای چاپ‌شدهٔ `0.000` و `1.000` گرد شده‌اند و الزاماً دقیقاً صفر یا یک نیستند.

در BCE، **۱۳۳ تصویر** هیچ score بالاتر از `0.5`، **۶۹۸ تصویر** دقیقاً یک score بالاتر از `0.5`، و **۳۶ تصویر** بیش از یک score بالاتر از `0.5` داشتند. با این وجود، برای گزارش multiclass در همهٔ تصاویر `argmax(logits)` یک کلاس انتخاب می‌کند؛ حتی اگر هیچ sigmoidی از `0.5` عبور نکند. برای مثال در تصویر شمارهٔ ۲۰ با برچسب واقعی کامیون، CE حدود `0.855` برای کامیون و `0.145` برای کامیونت داد، اما BCE برای هر دو کلاس مقدارهای نزدیک ۱ (`1.000` و `0.991` پس از گردکردن) داد. این مثال استقلال تصمیم‌های BCE را نشان می‌دهد.

| وضعیت روی validation | تعداد تصاویر |
|---|---:|
| هر دو درست | ۷۱۷ |
| فقط CE درست | ۲۶ |
| فقط BCE درست | ۲۴ |
| هر دو غلط | ۱۰۰ |

در مجموع پیش‌بینی کلاس دو مدل برای **۵۹ تصویر** متفاوت بود. از این ۵۹ مورد، ۵۰ مورد مربوط به درست‌بودن فقط یکی از مدل‌هاست و ۹ مورد را هر دو غلط ولی به دو کلاس متفاوت پیش‌بینی کرده‌اند. بنابراین اختلاف دقت نهایی فقط **۲ تصویر** است، اما اختلاف رفتاری مدل‌ها از آن بیشتر است.

میانگین بیشترین softmax مدل CE روی تصاویر درست/غلط به‌ترتیب `0.972`/`0.871` بود؛ بنابراین خطاهای آن هم اغلب score بالایی دارند. میانگین بیشترین sigmoid مدل BCE روی تصاویر درست/غلط `0.890`/`0.522` بود. این دو نوع score را مستقیم از نظر «اطمینان» رتبه‌بندی نمی‌کنیم: sigmoidهای BCE مستقل‌اند و حتی softmax بالا نیز به‌تنهایی تضمین نمی‌کند مدل کالیبره است. برای خروجی نهایی تک‌برچسبی، checkpoint منتخب CrossEntropy مناسب‌تر است و BCE در گزارش آموزشی می‌ماند.


## CNN، قدم ۵۷: ترکیب روش‌های تنظیم آموزش

تا اینجا augmentation، dropout، weight decay و scheduler را **جداگانه** آزمودیم. حالا یک مدل CNN تازه را با ترکیب این چهار مورد آموزش می‌دهیم: `RandomHorizontalFlip(p=0.5)` فقط روی training، `Dropout(p=0.3)`، `Adam(weight_decay=1e-4)` و `ReduceLROnPlateau` با `factor=0.5` و `patience=2` بر اساس validation loss. `MaxPool`، معماری دو کانولوشنی، split، batch size=32، learning rate اولیهٔ `0.001`، seed=42 و بودجهٔ ۳۰ epoch ثابت می‌مانند.

این آزمایش **ترکیبی** است؛ اگر نتیجه بهتر یا بدتر شود، آن را به اثر مجموعهٔ تنظیم‌ها نسبت می‌دهیم، نه به یکی از چهار عامل به‌تنهایی. حتی اگر هر عامل جداگانه دقت را بهتر نکرده باشد، تعامل آن‌ها ممکن است نتیجهٔ متفاوتی بدهد؛ تضمینی هم برای بهبود وجود ندارد. مانند آزمایش‌های قبلی، checkpoint از بین ۳۰ epoch با **بیشترین validation accuracy** انتخاب می‌شود؛ scheduler فقط برای تغییر learning rate از validation loss استفاده می‌کند.

سلول بعدی فایل runner را reload می‌کند تا تغییرات جدید در کرنل اعمال شوند، سپس یک آموزش جدید انجام می‌دهد و accuracy، macro-F1، بهترین epoch و تغییرات learning rate را با اجراهای پایه و تکی مقایسه می‌کند.


In [ ]:
import importlib
import json

experiment_runner = importlib.reload(experiment_runner)
experiment_runner.run("regularized", epochs=30)

comparison_results = json.loads(
    (project_root / "reports" / "experiment_results.json").read_text()
)
regularized = comparison_results["regularized"]
assert regularized["training_images"] == 3469
assert regularized["validation_images"] == 867
assert regularized["epochs"] == 30
assert regularized["configuration"]["dropout_p"] == 0.3
assert regularized["configuration"]["weight_decay"] == 1e-4
assert regularized["configuration"]["scheduler"]["name"] == "ReduceLROnPlateau"
assert "RandomHorizontalFlip" in regularized["configuration"]["augmentation"]

experiment_names = (
    "cnn_baseline", "augment", "dropout", "weight_decay", "scheduler", "regularized"
)
assert len({comparison_results[name]["split_fingerprint"] for name in experiment_names}) == 1
for name in experiment_names:
    result = comparison_results[name]
    macro_f1 = result.get("macro_f1", result.get("macro avg", {}).get("f1-score"))
    print(f'{name:15s} | accuracy {result["accuracy"]:.1%} | '
          f'macro-F1 {macro_f1:.3f} | best epoch {result["best_epoch"]}')

previous_lr = None
for row in regularized["history"]:
    lr = row["learning_rates"][0]
    if lr != previous_lr:
        print(f'Combined model: epoch {row["epoch"]:02d} starts with learning rate {lr:.6f}')
        previous_lr = lr


## CNN، قدم ۵۸: نتیجهٔ ترکیب روش‌ها

مدل ترکیبی روی همان **۳۴۶۹ تصویر training** و **۸۶۷ تصویر validation** برای ۳۰ epoch آموزش دید. ترکیب شامل flip آموزشی، dropout=`0.3`، Adam با weight decay=`1e-4` و scheduler مبتنی بر validation loss بود؛ معماری CNN و CrossEntropy همان مرجع بودند.

| اجرا | بهترین epoch بر اساس accuracy | پاسخ درست / ۸۶۷ | accuracy | macro-F1 | validation loss در checkpoint منتخب |
|---|---:|---:|---:|---:|---:|
| CNN پایه | ۲۴ | ۷۴۳ | ۸۵٫۷٪ | ۰٫۸۵۷ | `0.9704` |
| فقط scheduler | ۲۴ | ۷۴۳ | ۸۵٫۷٪ | ۰٫۸۵۷ | `0.7738` |
| ترکیب چهار روش | ۱۴ | ۷۴۰ | ۸۵٫۴٪ | ۰٫۸۵۱ | `0.6201` |

پس ترکیب روش‌ها در **accuracy و macro-F1 کلی** از مدل پایه بهتر نشد و ۳ پاسخ درست کمتر داشت. با این حال validation loss در checkpoint منتخب پایین‌تر است. این‌بار هر دو مدل CrossEntropy دارند و روی یک validation ارزیابی شده‌اند، پس عدد loss قابل مقایسه است؛ اما loss کمتر لزوماً تعداد خطاهای طبقه‌بندی را کمتر نمی‌کند. کمترین validation loss کل اجرای ترکیبی حدود `0.5835` در epoch ۱۰ بود، با accuracy برابر `84.4%`؛ checkpoint منتخبِ بیشترین accuracy از epoch ۱۴ است.

اثر کلاسی یکسان نبود: recall وانت از `85.0%` به `91.9%` بهتر شد، اما recall سواری از `83.6%` به `77.3%` و کامیونت از `77.5%` به `72.5%` افت کرد. پس پایین‌تر بودن loss یا بهترشدن یک کلاس، به‌تنهایی برای انتخاب این مدل به‌عنوان بهترین CNN کافی نیست. در پایان آموزش، train accuracy نزدیک `99.7%` ولی validation accuracy حدود `84.8%` بود؛ شکاف تعمیم همچنان وجود داشت.

scheduler در پایان epoch ۸ نخستین کاهش learning rate را انجام داد، بنابراین آموزش epoch ۹ با `0.0005` آغاز شد. بهترین checkpoint در epoch ۱۴ با learning rate برابر `0.00025` به‌دست آمد. نتیجهٔ این آزمایش فقط دربارهٔ **همین ترکیب مشخص** است؛ از آن نمی‌توان نتیجه گرفت هر نوع ترکیب تنظیم‌ها نامفید است.
